# 1 Descargar Ventas y Visitas (aje-prd-analytics-artifacts-s3)

In [1]:
# import psutil
# import time
# import threading
# import matplotlib.pyplot as plt

# # Variables globales
# mem_usage_gb = []
# mem_usage_percent = []
# cpu_usage_gb = []
# cpu_usage_percent = []
# timestamps = []
# running = True  # Variable de control

# # Función de monitoreo en segundo plano
# def monitor_resources():
#     global running
#     start_time = time.time()
    
#     while running:
#         # Obtener información del sistema
#         mem_info = psutil.virtual_memory()
#         cpu_percent = psutil.cpu_percent(interval=0.5)  # Medir CPU sin bloquear
#         cpu_used_gb = (cpu_percent / 100) * (psutil.virtual_memory().total / (1024**3))  # Estimación del uso de CPU en GB
        
#         # Guardar datos
#         timestamp = time.time() - start_time
#         mem_usage_gb.append(mem_info.used / (1024 ** 3))  # RAM usada en GB
#         mem_usage_percent.append(mem_info.percent)  # RAM en %
#         cpu_usage_gb.append(cpu_used_gb)  # CPU en GB
#         cpu_usage_percent.append(cpu_percent)  # CPU en %
#         timestamps.append(timestamp)
        
#         # Imprimir valores en tiempo real
#         # print(f"Tiempo: {timestamp:.2f}s | RAM: {mem_info.percent:.2f}% ({mem_usage_gb[-1]:.2f} GB) | CPU: {cpu_percent:.2f}% ({cpu_used_gb:.2f} GB)")
        
#         time.sleep(1)  # Medir cada 1 segundo

# # Iniciar monitoreo en segundo plano
# monitor_thread = threading.Thread(target=monitor_resources, daemon=True)
# monitor_thread.start()

# print("Monitoreo de CPU y RAM iniciado en segundo plano...")

In [2]:
!pip install --upgrade pip setuptools wheel
!pip install --upgrade pyarrow==17.0.0
!pip install "awswrangler[redshift]" --no-build-isolation
!pip install psycopg2-binary

In [3]:
import pandas as pd
import redshift_connector
import awswrangler as wr
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import os
import boto3
import io
import pytz
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

In [4]:
def comprobar_inputs():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"
    
    hoy = datetime.now(pytz.timezone("America/Lima")).date()
    errores = []

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" not in objetos:
        print("ERROR: No se encontraron archivos en la ruta especificada.")
        return

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        key = objeto["Key"]
        if key == "pedido_sugerido/data-v1/cam/ventas_sellout_cam000":
            print(f"OJO: El key {key} no aplica a esta validacion.")
            continue
        print(key)
        # Omitir "carpetas" en S3
        if key.endswith("/"):
            continue

        last_modified = objeto["LastModified"].date()
        size_kb = objeto["Size"] / 1024  # Convertir a KB

        # Verificar si el objeto tiene contenido real (FULL_OBJECT)
        if objeto["Size"] == 0:
            errores.append(f"ERROR: El archivo {key} está vacío.")

        # Verificar si el archivo ha sido modificado hoy
        if last_modified != hoy:
            errores.append(f"ERROR: El archivo {key} no ha sido modificado hoy ({hoy}), su ultima fecha de modificacion fue {last_modified}.")

        # Verificar si el tamaño del archivo es menor a 1 KB
        if size_kb < 1:
            errores.append(f"ALERTA: El archivo {key} tiene un tamaño menor a 1 KB ({size_kb:.2f} KB).")

    # Mostrar los errores y lanzar una excepción si es necesario
    if errores:
        for error in errores:
            print(error)
        raise ValueError("Se encontraron problemas con los archivos en S3.")
    else:
        print("Todo bien :D")

In [5]:
comprobar_inputs()

pedido_sugerido/data-v1/cam/
pedido_sugerido/data-v1/cam/D_stock_cr.csv
pedido_sugerido/data-v1/cam/D_stock_gt.csv
pedido_sugerido/data-v1/cam/D_stock_ni.csv
pedido_sugerido/data-v1/cam/D_stock_pa.csv
pedido_sugerido/data-v1/cam/maestro_productos_cam000
pedido_sugerido/data-v1/cam/ventas_cam000
pedido_sugerido/data-v1/cam/ventas_cam001
OJO: El key pedido_sugerido/data-v1/cam/ventas_sellout_cam000 no aplica a esta validacion.
pedido_sugerido/data-v1/cam/visitas_cam000
Todo bien :D


In [6]:
# Obtener cod SKUs y descripcion de CAM
query=f'''select * from
    comercial_cam.dim_producto
    where estado='A' and cod_compania = '0096';
    '''

con = wr.data_api.redshift.connect(cluster_id = 'dwh-cloud-storage-salesforce-prod', database = 'dwh_prod',db_user='dwhuser')
maestro_prod = wr.data_api.rds.read_sql_query( query, con)

In [7]:
maestro_prod = maestro_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
maestro_prod = maestro_prod.groupby(["cod_articulo_magic"]).first().reset_index()
maestro_prod.to_csv("Input/D_pan_masterProd.csv", index=False)

In [8]:
rutas_test = [
11101,11102,11103,11104,11105,11201,11202,11203,11204,11205,11206,11207,
11301,11302,11303,11401,11402,11403,11404,11405,11701,11702,11703,11704,
11705,11706,11707,11708,11501,11502,11503,11504,11601,11602,11603,11604,
11801,11802,11803,11901,11902,11101,11102,11103,11104,11105,11201,11202,
11203,11204,11205,11206,11301,11302,11303,11401,11402,11403,11404,11405,
11701,11702,11703,11704,11705,11706,11707,11708,11501,11502,11503,11504,
11601,11602,11603,11604,11801,11802,11803,11901,11902]


In [9]:
len(rutas_test)

81

In [10]:
rutas_test = list(set(rutas_test)) # quitamos rutas duplicadas
len(rutas_test)

41

In [11]:
clientes_ruta_test = []

In [12]:
def descargar_visitas():
    # Guardar clientes de las rutas
    global clientes_ruta_test

    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0 and objeto["Key"].split("/")[-1] == "visitas_cam000":
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "visitas_cam000":
                print(objeto)
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df.pais__c == "PA")
                    & (df["cod_ruta"].isin(rutas_test))
                    & (df.codigo_canal__c == 2)
                ].reset_index(drop=True)
                
                # Guardamos estos clientes para descargarlos en ventas (por si cambiaron de ruta)
                clientes_ruta_test = df["codigo_cliente__c"].unique()
                nombre_csv = f"Input/{nombre_archivo}.parquet"
                df["personalizacion"] = df["personalizacion"].astype(str)
                df.to_parquet(nombre_csv, index=False)
            else:
                continue

In [13]:
%%time
descargar_visitas()

{'Key': 'pedido_sugerido/data-v1/cam/visitas_cam000', 'LastModified': datetime.datetime(2026, 3, 30, 15, 2, 28, tzinfo=tzlocal()), 'ETag': '"957996d65c2604d1523a716d29797013-1"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 43502592, 'StorageClass': 'STANDARD'}
CPU times: user 847 ms, sys: 189 ms, total: 1.04 s
Wall time: 1.43 s


In [14]:
def descargar_ventas():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)

    # Creamos DF que juntará los 2 archivos
    final_csv = pd.DataFrame()

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0:
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "ventas_cam000":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df["cod_ruta"].isin(rutas_test))
                        | (df["cod_cliente"].isin(clientes_ruta_test))
                ].reset_index(drop=True)[['id_cliente', 'id_sucursal', 'id_producto',
       'fecha_liquidacion',"cod_ruta","cod_modulo",
       'cod_zona', 'cant_cajafisicavta', 'cant_cajaunitvta','imp_netovta',
       'cod_compania', 'desc_compania', 'cod_sucursal',
       'desc_sucursal', 'cod_pais', 'fecha_creacion_cliente', 'cod_cliente',
       'desc_marca', 'desc_formato', 'desc_categoria', 'cod_giro',
       'cod_subgiro', 'desc_giro', 'desc_subgiro', 'fecha_proceso']]
                final_csv = pd.concat([final_csv, df], ignore_index=True)
            elif nombre_archivo == "ventas_cam001":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df["cod_ruta"].isin(rutas_test))
                        | (df["cod_cliente"].isin(clientes_ruta_test))
                ].reset_index(drop=True)[['id_cliente', 'id_sucursal', 'id_producto',
       'fecha_liquidacion',"cod_ruta","cod_modulo",
       'cod_zona', 'cant_cajafisicavta', 'cant_cajaunitvta','imp_netovta',
       'cod_compania', 'desc_compania', 'cod_sucursal',
       'desc_sucursal', 'cod_pais', 'fecha_creacion_cliente', 'cod_cliente',
       'desc_marca', 'desc_formato', 'desc_categoria', 'cod_giro',
       'cod_subgiro', 'desc_giro', 'desc_subgiro', 'fecha_proceso']]
                final_csv = pd.concat([final_csv, df], ignore_index=True)

            else:
                continue

    nombre_csv = f"Input/ventas_cam_all.parquet"
    final_csv.to_parquet(nombre_csv, index=False)

In [15]:
%%time
descargar_ventas()

{'ResponseMetadata': {'RequestId': 'EDP0J98GM5KFVSZK', 'HostId': 'xOqAjKP0vTwrqWx4CZcTOUeMbBBUGhosDHkV6Sd/taFbM3W1c0z0xI/j2o2TKuD7g8hW4jAi8x+cVD2ytMCq/Mr2kqJJiHOi', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'xOqAjKP0vTwrqWx4CZcTOUeMbBBUGhosDHkV6Sd/taFbM3W1c0z0xI/j2o2TKuD7g8hW4jAi8x+cVD2ytMCq/Mr2kqJJiHOi', 'x-amz-request-id': 'EDP0J98GM5KFVSZK', 'date': 'Mon, 30 Mar 2026 21:40:50 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'pedido_sugerido/data-v1/cam/', 'LastModified': datetime.datetime(2024, 2, 8, 8, 33, 15, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'pedido_sugerido/data-v1/cam/D_stock_cr.csv', 'LastModified': datetime.datetime(2026, 3, 30, 14, 3, 37, tzinfo=tzlocal()), 'ETag': '"7dda2ed19935555f934134f38e064ffd"', 'ChecksumAlgorithm': ['CRC32'], 'Checksu

/tmp/ipykernel_30399/2670914792.py:28: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(content), sep=";")


CPU times: user 2min 3s, sys: 24.3 s, total: 2min 28s
Wall time: 4min 20s


## 1.1 Juntar VENTAS y VISITAS

In [16]:
import pandas as pd
from datetime import datetime, timedelta

In [17]:
pan_ventas = pd.read_parquet("Input/ventas_cam_all.parquet")
pan_visitas = pd.read_parquet("Input/visitas_cam000.parquet")

In [18]:
pan_ventas = pan_ventas[pan_ventas.cod_pais=="PA"].reset_index(drop=True)

In [19]:
pan_visitas = pan_visitas[(pan_visitas.codigo_canal__c==2)&(pan_visitas.compania__c=="0096")].reset_index(drop=True)

In [20]:
pan_ventas["cod_articulo_magic"] = pan_ventas["id_producto"].str.split("|").str[-1]
pan_ventas["cod_articulo_magic"] = pan_ventas["cod_articulo_magic"].astype(int)

In [21]:
pan_ventas.shape

(1420463, 26)

In [22]:
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print(hoy)
# Filtrar solo las filas con la fecha de hoy
pan_ventas = pan_ventas[pan_ventas['fecha_proceso'] == hoy]

20260330


In [23]:
pan_ventas.shape

(1420463, 26)

In [24]:
# Obtener la fecha y año actual
current_date = datetime.now(pytz.timezone("America/Lima"))
# Formatear la fecha en el formato "YYYYMM"
formatted_date = current_date.strftime("%Y-%m-%d")
formatted_date

'2026-03-30'

In [25]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Panama/Input/visitas_cam000_{formatted_date}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(pan_visitas, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Panama/Input/visitas_cam000_2026-03-30.csv'],
 'partitions_values': {}}

In [26]:
PS_VAL_total_skus_venta = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_venta

65

In [27]:
# PAN VENTAS
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_ventas["id_cliente"] = (
    "CAM"
    + "|"
    + pan_ventas["cod_compania"].astype(str)
    + "|"
    + pan_ventas["cod_cliente"].astype(str)
)

# PAN VISITAS
pan_visitas["compania__c"] = (
    pan_visitas["compania__c"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_visitas["id_cliente"] = (
    "CAM"
    + "|"
    + pan_visitas["compania__c"].astype(str)
    + "|"
    + pan_visitas["codigo_cliente__c"].astype(str)
)

In [28]:
PS_VAL_ultima_venta = pan_ventas["fecha_liquidacion"].sort_values().unique()[-1:]
PS_VAL_ultima_venta

array(['2026-03-28'], dtype=object)

In [29]:
PS_VAL_ultima_visita = pan_visitas["ultima_visita"].sort_values().unique()[-2:-1]
PS_VAL_ultima_visita

array(['2026-03-28'], dtype=object)

In [30]:
PS_VAL_total_clientes_venta = pan_ventas["id_cliente"].nunique()
PS_VAL_total_clientes_venta

13105

In [31]:
PS_VAL_total_clientes_visita = pan_visitas["id_cliente"].nunique()
PS_VAL_total_clientes_visita

12453

In [32]:
pan_visitas = pan_visitas[pan_visitas.codigo_canal__c == 2].reset_index(drop=True)

In [33]:
PS_VAL_total_clientes_visita_trad = pan_visitas["id_cliente"].nunique()
PS_VAL_total_clientes_visita_trad

12453

In [34]:
# Eliminar Duplicados y quedarse con la ultima visita más reciente
visita_default_semana_atras = (datetime.now(pytz.timezone("America/Lima")) - timedelta(days=7)).strftime("%Y-%m-%d")
pan_visitas["ultima_visita"] = pan_visitas["ultima_visita"].fillna(
    visita_default_semana_atras
)
pan_visitas = (
    pan_visitas.sort_values(["id_cliente", "ultima_visita"], ascending=False)
    .groupby("id_cliente")
    .head(1)
)

In [35]:
# Quitamos cod_modulo de ventas para luego quedarnos con el de visitas
# pan_ventas = pan_ventas.drop(["cod_modulo","cod_ruta"], axis=1)

In [36]:
# pan_ventas=pd.merge(pan_ventas,pan_visitas[["id_cliente","dias_de_visita__c","periodo_de_visita__c","ultima_visita","cod_ruta","cod_modulo"]],on=["id_cliente"],how="outer").drop_duplicates()

In [37]:
# # Combina ventas y visitas (nos quedamos con ruta y modulo de visitas) si es nulo, nos quedamos con el valor de ventas
# # Combinar ambos DataFrames por 'id_cliente'
# df_merged = pd.merge(pan_ventas, pan_visitas[["id_cliente","dias_de_visita__c","periodo_de_visita__c","ultima_visita","cod_ruta","cod_modulo"]], on='id_cliente', how='outer', suffixes=('_df1', '_df2'))

# # Reemplazar los valores nulos de 'cod_ruta_df2' con los valores de 'cod_ruta_df1'
# df_merged['cod_ruta'] = df_merged['cod_ruta_df2'].combine_first(df_merged['cod_ruta_df1'])
# df_merged['cod_modulo'] = df_merged['cod_modulo_df2'].combine_first(df_merged['cod_modulo_df1'])

# # Eliminar las columnas innecesarias
# df_final = df_merged.drop(columns=['cod_ruta_df1', 'cod_ruta_df2'])
# df_final = df_final.drop(columns=['cod_modulo_df1', 'cod_modulo_df2'])

# Combina ventas y visitas (nos quedamos con ruta y módulo de visitas si es nulo, usamos el valor de ventas)
df_merged = pd.merge(
    pan_ventas,
    pan_visitas[
        [
            "id_cliente",
            "dias_de_visita__c",
            "periodo_de_visita__c",
            "ultima_visita",
            "cod_ruta",
            "cod_modulo",
            "eje_potencial__c",
        ]
    ],
    on="id_cliente",
    how="inner",  # Este es un merge 'outer', para conservar todos los clientes de ambos DataFrames.
    suffixes=(
        "_df1",
        "_df2",
    ),  # Añadimos sufijos para diferenciar las columnas que se solapan
)

# Reemplazar los valores nulos de 'cod_ruta_df2' con los valores de 'cod_ruta_df1', y viceversa
# Si 'cod_ruta_df2' es nulo, se usará el valor de 'cod_ruta_df1'. Si ambos son nulos, se mantendrá nulo.
df_merged["cod_ruta"] = df_merged["cod_ruta_df2"].combine_first(
    df_merged["cod_ruta_df1"]
)
df_merged["cod_modulo"] = df_merged["cod_modulo_df2"].combine_first(
    df_merged["cod_modulo_df1"]
)

# Eliminar las columnas innecesarias ('cod_ruta_df1', 'cod_ruta_df2', 'cod_modulo_df1', 'cod_modulo_df2')
df_merged = df_merged.drop(
    columns=["cod_ruta_df1", "cod_ruta_df2", "cod_modulo_df1", "cod_modulo_df2"]
)

# df_final ahora contiene todos los clientes de ambos DataFrames, con las columnas combinadas de manera adecuada.

In [38]:
pan_ventas = df_merged.copy()

In [39]:
pan_ventas["cod_cliente"] = pan_ventas["cod_cliente"].astype(str)
pan_ventas["cod_ruta"] = pan_ventas["cod_ruta"].astype(int)
pan_ventas["cod_modulo"] = pan_ventas["cod_modulo"].astype(int)

In [40]:
# Al cruzar visita con ventas. hay clientes de. visita que no tienen informacion de venta, esto se remplaza a nivel de ruta.
# si la ruta entera no tiene ventas, se ignora
# Este df guarda clientes con visitas y sin ventas para luego recomendarlos por separado
df_clientes_con_visitas_sin_ventas = pd.DataFrame()
for ruta in pan_ventas["cod_ruta"].unique():
    temp_df = pan_ventas[pan_ventas["cod_ruta"] == ruta]
    # Si la ruta entera no tiene ninguna venta, se guarda en un dataframe aparte
    if len(temp_df) == (temp_df["cod_articulo_magic"].isnull().sum()):
        df_clientes_con_visitas_sin_ventas = pd.concat(
            [df_clientes_con_visitas_sin_ventas, temp_df], axis=0
        ).reset_index(drop=True)

In [41]:
pan_ventas["id_cliente"].nunique()

12250

In [42]:
PS_VAL_total_clientes_merged = pan_ventas["id_cliente"].nunique()
PS_VAL_total_clientes_merged

12250

In [43]:
PS_VAL_total_skus_merged = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_merged

65

In [44]:
PS_VAL_total_fechas_unicas = pan_ventas.fecha_liquidacion.sort_values().nunique()
PS_VAL_total_fechas_unicas

348

# 2 Ajustar DF de clientes: Se filtrará a los clientes que tienen visita al día siguiente

In [45]:
# CAMBIAR DEPORADE POR SPORADE
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].str.strip()
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].replace({"DEPORADE": "SPORADE"})

pan_prod = pd.read_csv("Input/D_pan_masterProd.csv")

# PAN SEGMENTOS
pan_ventas = pan_ventas.rename(columns={"eje_potencial__c": "new_segment"})
# mapear valores
mapping = {
    "S1": "BLINDAR",
    "S2": "DESARROLLAR",
    "S4": "MANTENER",
    "S5": "OPTIMIZAR",
}

pan_ventas["new_segment"] = pan_ventas["new_segment"].map(mapping)
pan_ventas["new_segment"] = pan_ventas["new_segment"].fillna("OPTIMIZAR")

In [46]:
pan_ventas = pd.merge(
    pan_ventas,
    pan_prod[["cod_articulo_magic", "desc_articulo"]],
    how="left",
    on="cod_articulo_magic",
)

pan_ventas.rename(columns={"fecha_creacion_cliente": "fecha_creacion"}, inplace=True)
pan_ventas["mes"] = pd.to_datetime(pan_ventas["fecha_liquidacion"]).dt.strftime(
    "%Y-%m-01"
)

In [47]:
pan_ventas.to_parquet("Processed/ventas_panama_12m.parquet", index=False)

## 2.3 Seleccionando SKUs que no se commpran en los ultimos 3 dias
La variable **sucursal_sku** contiene todos los SKUs disponibles por sucursal

In [48]:
# pan_ventas[["cod_compania","cod_sucursal"]].drop_duplicates().to_csv("PA_compa_sucursal_2025-07-31.csv",index=False)

In [49]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [50]:
sucursales = pan_ventas["cod_sucursal"].unique()
sucursales

array([1, 3, 2, 4, 5])

In [51]:
# Obtener la fecha actual
fecha_actual = datetime.now(pytz.timezone("America/Lima")).date()
# Lista para almacenar las fechas
last_3_days = []
dias_atras = 3
sucursal_sku = {}
# Obtener las fechas de los últimos 3 días
for i in range(1, dias_atras + 1):
    fecha = fecha_actual - timedelta(days=i)
    fecha_formato = fecha.strftime("%Y-%m-%d")
    last_3_days.append(fecha_formato)

In [52]:
last_3_days

['2026-03-29', '2026-03-28', '2026-03-27']

In [53]:
for sucursal in sucursales:
    temp = pan_ventas[pan_ventas["cod_sucursal"] == sucursal]
    # sku_fecha=temp[["cod_articulo_magic","fecha_liquidacion"]].sort_values(["cod_articulo_magic","fecha_liquidacion"]).drop_duplicates().reset_index(drop=True)
    # sku_fecha=sku_fecha.groupby("cod_articulo_magic")[["cod_articulo_magic","fecha_liquidacion"]].tail(3)
    # #Listando SKUs comprados al menos una vez en los ultimos 3 dias
    # sku_in_last_3_days=sku_fecha[sku_fecha.fecha_liquidacion.isin(last_3_days)]["cod_articulo_magic"].unique()
    # print(f"Sucursal {sucursal} eliminando SKUs:",sku_fecha[(~sku_fecha["cod_articulo_magic"].isin(sku_in_last_3_days))].reset_index(drop=True)["cod_articulo_magic"].sort_values().unique())
    # #SKU COMPRADO AL MENOS una vez en los ultimos 3 dias
    # sku_fecha=sku_fecha[sku_fecha["cod_articulo_magic"].isin(sku_in_last_3_days)].reset_index(drop=True)
    # temp=temp[temp["cod_articulo_magic"].isin(sku_fecha["cod_articulo_magic"].unique())].reset_index(drop=True)
    sucursal_sku[sucursal] = temp["cod_articulo_magic"].sort_values().unique()

## 2.4 Filtrar clientes a visitar mañana

In [54]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [55]:
# Filtrar ventas mayores a 0
# pan_ventas=pan_ventas[pan_ventas["imp_netovta"]>0]

# En lugar de omitir estos clientes, mejor los consideramos como OPTIMIZAR
# pan_ventas.loc[pan_ventas["imp_netovta"] <= 0, "new_segment"] = "OPTIMIZAR"

In [56]:
PS_VAL_total_clientes_imp_netovta_positivo = pan_ventas.id_cliente.nunique()
PS_VAL_total_clientes_imp_netovta_positivo

12250

In [57]:
pan_ventas.dias_de_visita__c.unique()

array(['3;6', '2', '5', '1', '6', '1;4', '3', '4', '2;5', '7'],
      dtype=object)

In [58]:
pan_ventas.dias_de_visita__c.value_counts(dropna=False)

dias_de_visita__c
1      212165
4      187794
6      187107
5      183740
2      179287
3      175820
2;5     91467
3;6     87363
1;4     67919
7         390
Name: count, dtype: int64

In [59]:
pan_visitas.groupby("dias_de_visita__c").id_cliente.nunique()

dias_de_visita__c
1      1968
1;4     281
2      1890
2;5     399
3      1870
3;6     388
4      1892
4;7       1
5      1860
6      1901
7         3
Name: id_cliente, dtype: int64

In [60]:
data_test = (
    pan_ventas[
        ["id_cliente", "dias_de_visita__c", "periodo_de_visita__c", "ultima_visita"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
data_test["ultima_visita"] = pd.to_datetime(
    data_test["ultima_visita"], format="%Y-%m-%d"
)
# Calcula la diferencia en días entre la fecha de cada fila y la fecha de hoy
fecha_actual = datetime.now() + timedelta(days=0)
data_test["dias_pasados"] = (fecha_actual - data_test["ultima_visita"]).dt.days
fecha_actual

datetime.datetime(2026, 3, 30, 21, 45, 34, 113549)

In [61]:
# Obtener el día de la semana actual (1 para lunes, 2 para martes, ..., 7 para domingo)
dia_actual = datetime.now(pytz.timezone("America/Lima")).weekday() + 1

# Si hoy es domingo (7), el día siguiente es lunes (1), de lo contrario, es el siguiente día al actual
if dia_actual == 6:
    dia_siguiente = 7
else:
    dia_siguiente = (dia_actual + 1) % 7
    # dia_siguiente = (dia_actual ) % 7
    #dia_pasado_manana = (dia_siguiente % 7) + 1

# Filtrar los clientes que serán visitados mañana
clientes_a_visitar_manana = data_test[
    data_test["dias_de_visita__c"]
    .astype(str)
    .apply(lambda x: str(dia_siguiente) in x.split(";"))
    #.apply(lambda x: str(dia_siguiente) in x.split(";") or str(dia_pasado_manana) in x.split(";"))
].reset_index(drop=True)
print("dia_actual", dia_actual)
print("dia_siguiente", dia_siguiente)
#print("dia_siguiente", dia_pasado_manana)

dia_actual 1
dia_siguiente 2


In [62]:
# Definir condiciones de filtro
condicion_f1 = clientes_a_visitar_manana["periodo_de_visita__c"] == "F1"
condicion_f2 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F2") & (
    clientes_a_visitar_manana["dias_pasados"] > 13
)
condicion_f3 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F3") & (
    clientes_a_visitar_manana["dias_pasados"] > 20
)
condicion_f4 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F4") & (
    clientes_a_visitar_manana["dias_pasados"] > 27
)

# Aplicar las condiciones de filtro
clientes_a_visitar_manana = clientes_a_visitar_manana[
    condicion_f1 | condicion_f2 | condicion_f3 | condicion_f4
].reset_index(drop=True)

In [63]:
clientes_a_visitar_manana.isnull().sum()

id_cliente              0
dias_de_visita__c       0
periodo_de_visita__c    0
ultima_visita           0
dias_pasados            0
dtype: int64

In [64]:
clientes_a_visitar_manana.periodo_de_visita__c.unique()

array(['F1'], dtype=object)

In [65]:
clientes_a_visitar_manana.dias_de_visita__c.unique()

array(['2', '2;5'], dtype=object)

In [66]:
pan_ventas = pan_ventas[
    pan_ventas["id_cliente"].isin(clientes_a_visitar_manana["id_cliente"])
].reset_index(drop=True)

In [67]:
pan_ventas.periodo_de_visita__c.unique()

array(['F1'], dtype=object)

In [68]:
pan_ventas.dias_de_visita__c.unique()

array(['2', '2;5'], dtype=object)

In [69]:
PS_VAL_total_clientes_para_manhana = pan_ventas.id_cliente.nunique()
PS_VAL_total_clientes_para_manhana

2247

In [70]:
PS_VAL_total_skus_venta_para_manhana = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_venta_para_manhana

65

In [71]:
pan_ventas.groupby("cod_sucursal").cod_articulo_magic.nunique()

cod_sucursal
1    53
2    51
3    49
4    49
5    59
Name: cod_articulo_magic, dtype: int64

In [72]:
pan_ventas.to_parquet("Processed/pan_ventas_manana.parquet", index=False)

# 3. Pre Procesamiento

In [73]:
import pandas as pd
import numpy as np
from datetime import datetime

In [74]:
df_ventas = pd.read_parquet("Processed/pan_ventas_manana.parquet")

In [75]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(
    df_ventas["fecha_liquidacion"], format="%Y-%m-%d"
)
df_ventas["desc_marca"] = df_ventas["desc_marca"].str.strip()
df_ventas["desc_categoria"] = df_ventas["desc_categoria"].str.strip()

## 3.1 Mapeo de Pesos por Giro
Los pesos representan el orden en el que se recomendara la marca para cada cliente dependiendo del Giro

In [76]:
# Mapeo de pesos para ordenar las recomendaciones finales segun su importancia por giro
mapeo_pesos = {}
mapeo_diccionario = {}

In [77]:
for giro_v in df_ventas["desc_subgiro"].unique():
    temp = df_ventas[(df_ventas["desc_subgiro"] == giro_v)]
    ranks = temp.groupby("desc_categoria")["cant_cajafisicavta"].sum().reset_index()
    ranks.columns = ["index","desc_categoria"]
    ranks = ranks.sort_values(by="desc_categoria", ascending=False)
    if len(list(ranks["desc_categoria"])) <= 5:
        ranks["Ranking"] = range(1, len(ranks) + 1)
    elif len(list(ranks["desc_categoria"])) > 5:
        a = list(ranks["desc_categoria"])
        b = [1, 1, 2, 2]
        # Calculamos el multiplicador para el mapeo de pesos segun la varianza
        if np.std(a) / np.mean(a) <= 1.2:
            multiplicador = 4
        else:
            multiplicador = 2
        if len(a) > 5:
            for i in range(4, len(a)):
                if a[3] <= a[i] * multiplicador:
                    b.append(3)
                else:
                    b.append(3 + i)
        ranks["Ranking"] = b

    print("*" * 20)
    print("GIRO: ", giro_v)
    print("Categorias: ", list(ranks["index"]))
    print("Counts: ", list(ranks["desc_categoria"]))
    print(ranks.set_index("index")["Ranking"].to_dict())
    mapeo_diccionario[giro_v] = ranks.set_index("index")["Ranking"].to_dict()

********************
GIRO:  RESTAURANT TRADICIONAL
Categorias:  ['GASEOSAS', 'AGUA', 'JUGOS LIGEROS', 'ACEITE', 'BEBIDAS DEPORTIVAS', 'ENERGIZANTE', 'SUERO', 'SOPAS', 'NECTAR', 'DETERGENTE']
Counts:  [11560.583333333334, 6207.333333333334, 4566.0, 1341.0, 846.0, 821.0, 175.0, 66.0, 28.0, 22.4999999999991]
{'GASEOSAS': 1, 'AGUA': 1, 'JUGOS LIGEROS': 2, 'ACEITE': 2, 'BEBIDAS DEPORTIVAS': 3, 'ENERGIZANTE': 3, 'SUERO': 9, 'SOPAS': 10, 'NECTAR': 11, 'DETERGENTE': 12}
********************
GIRO:  VENTA EN CASA PARTICULAR
Categorias:  ['GASEOSAS', 'JUGOS LIGEROS', 'AGUA', 'ACEITE', 'ENERGIZANTE', 'BEBIDAS DEPORTIVAS', 'SUERO', 'SOPAS', 'NECTAR', 'DETERGENTE', 'PASTAS']
Counts:  [45863.541666666664, 8214.125, 2510.9333333333334, 2271.0, 685.1666666666669, 271.9166666666667, 245.0, 216.0, 105.0, 40.3333333333325, 2.0]
{'GASEOSAS': 1, 'JUGOS LIGEROS': 1, 'AGUA': 2, 'ACEITE': 2, 'ENERGIZANTE': 7, 'BEBIDAS DEPORTIVAS': 8, 'SUERO': 9, 'SOPAS': 10, 'NECTAR': 11, 'DETERGENTE': 12, 'PASTAS': 13}
******

## 3.2. Filtro de SKU por Ruta
Cada Ruta tiene una cantidad de SKUs distintos, la recomendacion será distinta para cada ruta, sin embargo, aquellas rutas que tengan pocos SKUs serán tratadas como una sola ruta

In [78]:
rutas = (
    df_ventas.groupby(["cod_ruta"])["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()["cod_ruta"]
    .unique()
)
rutas

array([11303, 11302, 11601, 11301, 11706, 11604, 11702, 11207, 11707,
       11502, 11801, 11705, 11204, 11206, 11202, 11104, 11704, 11602,
       11504, 11703, 11802, 11708, 11203, 11701, 11102, 11503, 11205,
       11103, 11401, 11404, 11101, 11201, 11402, 11501, 11901, 11105,
       11403, 11803, 11405, 11603, 11902])

In [79]:
low_sku_ruta = []

**Para rutas que tengan más de 45 SKUS**

In [80]:
for ruta in rutas:
    print("*" * 21)
    print("Ruta:", ruta)
    temp = df_ventas[(df_ventas["cod_ruta"] == ruta)]
    print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
    # print(f"Giros en ruta {ruta}:")
    # print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
    if temp["cod_articulo_magic"].nunique() <= 10:
        low_sku_ruta.append(ruta)
    else:
        temp.to_csv(f"Processed/rutas/D_{ruta}_ventas.csv", index=False)

*********************
Ruta: 11303
SKUs disponibles: 50
*********************
Ruta: 11302
SKUs disponibles: 53
*********************
Ruta: 11601
SKUs disponibles: 49
*********************
Ruta: 11301
SKUs disponibles: 50
*********************
Ruta: 11706
SKUs disponibles: 49
*********************
Ruta: 11604
SKUs disponibles: 49
*********************
Ruta: 11702
SKUs disponibles: 49
*********************
Ruta: 11207
SKUs disponibles: 50
*********************
Ruta: 11707
SKUs disponibles: 48
*********************
Ruta: 11502
SKUs disponibles: 49
*********************
Ruta: 11801
SKUs disponibles: 48
*********************
Ruta: 11705
SKUs disponibles: 48
*********************
Ruta: 11204
SKUs disponibles: 50
*********************
Ruta: 11206
SKUs disponibles: 49
*********************
Ruta: 11202
SKUs disponibles: 50
*********************
Ruta: 11104
SKUs disponibles: 48
*********************
Ruta: 11704
SKUs disponibles: 47
*********************
Ruta: 11602
SKUs disponibles: 49
**********

**Para rutas que tengan menos de 45 SKUS**

In [81]:
print("*" * 21)
print("Rutas:", low_sku_ruta)
temp = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
# print(f"Giros en ruta {ruta}:")
# print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
temp.to_csv("Processed/rutas/D_low_ruta_ventas.csv", index=False)

*********************
Rutas: []
SKUs disponibles: 0


# 4. Modelo Pedido Sugerido

## 4.1. Importando librerias necesarias

In [82]:
import os

from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType
from pyspark.sql.functions import when, col, regexp_replace, concat, countDistinct

from pyspark.ml.linalg import Vector
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [83]:
!java -version

openjdk version "11.0.30" 2026-01-20 LTS
OpenJDK Runtime Environment Corretto-11.0.30.7.1 (build 11.0.30+7-LTS)
OpenJDK 64-Bit Server VM Corretto-11.0.30.7.1 (build 11.0.30+7-LTS, mixed mode)


In [84]:
import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

## 4.2 Construccion del algoritmo ALS

In [85]:
# Se generan csv en Output/Panama/4-resultados/
def als_training_job(cedi, sku_len):

    spark = (
        SparkSession.builder.config("spark.memory.offHeap.enabled", "true")
        .config("spark.memory.offHeap.size", "10G")
        .appName("K-Means for ALS")
        .getOrCreate()
    )

    ventas = (
        spark.read.format("csv")
        .options(header=True, delimiter=",")
        .load(f"Processed/rutas/D_{cedi}_ventas.csv")
    )

    ventas = ventas.na.drop(subset=["fecha_liquidacion"])
    ventas = ventas.groupBy(
        ["id_cliente", "cod_articulo_magic", "cod_compania", "cod_cliente"]
    ).agg(countDistinct("fecha_liquidacion"))
    ventas = ventas.select(
        col("id_cliente").alias("id_cliente"),
        col("count(fecha_liquidacion)").alias("frecuencia"),
        col("cod_articulo_magic").alias("cod_articulo_magic"),
        col("cod_compania").alias("cod_compania"),
        col("cod_cliente").alias("cod_cliente"),
    )

    input_cols = ["frecuencia"]
    vec_assembler = VectorAssembler(inputCols=input_cols, outputCol="features")
    ventas_kmeans = vec_assembler.transform(ventas)
    ventas_kmeans = ventas_kmeans.fillna(0.0, subset=["frecuencia"])

    kmeans = KMeans(featuresCol="features", k=5)
    model = kmeans.fit(ventas_kmeans)

    resumen_ventas = model.transform(ventas_kmeans).groupby("prediction").max()
    resumen_ventas.orderBy(["max(frecuencia)"], ascending=0)
    resumen_ventas = resumen_ventas.select(
        col("prediction").alias("prediction"), col("max(frecuencia)").alias("max")
    )

    maxes = resumen_ventas.select("max").toPandas()["max"]
    maxes = maxes.astype(float)
    maxes = list(maxes)

    labels = [5, 4, 3, 2, 1]
    rank_map = dict(zip(labels, maxes))

    ventas = ventas.withColumn("frecuencia", col("frecuencia").cast(DoubleType()))
    ventas = ventas.withColumn(
        "rating",
        when(ventas["frecuencia"] < rank_map.get(labels[-1]), labels[-1])
        .when(ventas["frecuencia"] < rank_map.get(labels[-2]), labels[-2])
        .when(ventas["frecuencia"] < rank_map.get(labels[-3]), labels[-3])
        .when(ventas["frecuencia"] < rank_map.get(labels[-4]), labels[-4])
        .otherwise(labels[-5]),
    )

    cols = ["id_cliente", "cod_compania", "cod_cliente", "cod_articulo_magic", "rating"]
    als_records = ventas.select(*cols)

    als_records = als_records.withColumn(
        "clienteId", concat("cod_compania", "cod_cliente")
    )
    als_records = als_records.withColumn(
        "clienteId", col("clienteId").cast(IntegerType())
    )
    als_records = als_records.withColumn(
        "cod_articulo_magic", col("cod_articulo_magic").cast(IntegerType())
    )

    als_records = als_records.na.drop(subset=["clienteId", "cod_articulo_magic"])

    input_cols = ["ClienteId", "cod_articulo_magic", "rating"]
    als_records = als_records.select(*input_cols)

    als = ALS(
        rank=15,
        maxIter=5,
        implicitPrefs=True,
        ratingCol="rating",
        itemCol="cod_articulo_magic",
        userCol="ClienteId",
    )
    model = als.fit(als_records)

    # obtener matrices de descomposicion (usuario y articulos)
    user_matrix = model.userFactors
    item_matrix = model.itemFactors

    user_matrix_pd = user_matrix.toPandas()
    item_matrix_pd = item_matrix.toPandas()

    recs = model.recommendForAllUsers(sku_len)
    recs = recs.select("ClienteId", "recommendations.cod_articulo_magic")
    recs_to_parse = recs.toPandas()
    # print(recs_to_parse)

    lista_rec = [f"r{i+1}" for i in range(sku_len)]

    recs_to_parse[lista_rec] = pd.DataFrame(
        recs_to_parse["cod_articulo_magic"].tolist(), index=recs_to_parse.index
    )
    client_recs = pd.melt(
        recs_to_parse, id_vars=["ClienteId"], value_vars=lista_rec
    )

    client_recs["compania"] = client_recs["ClienteId"].apply(lambda x: str(x)[:2])
    client_recs["cliente"] = client_recs["ClienteId"].apply(lambda x: str(x)[2:])
    client_recs["compania"] = client_recs["compania"].apply(lambda x: x.rjust(4, "0"))
    client_recs["id_cliente"] = client_recs["compania"] + "|" + client_recs["cliente"]
    client_recs["cod_articulo_magic"] = client_recs["value"]

    client_recs = (
        client_recs[["id_cliente", "cod_articulo_magic"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Obtener el nombre del archivo CSV
    nombre_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo ya existe
    try:
        # Intentar abrir el archivo para lectura
        with open(nombre_archivo, "r") as f:
            # Si el archivo existe, header=False para no escribir cabeceras
            client_recs.to_csv(
                nombre_archivo, mode="a", header=False, index=False, sep=";"
            )
    except FileNotFoundError:
        # Si el archivo no existe, header=True para escribir las cabeceras
        client_recs.to_csv(nombre_archivo, mode="a", header=True, index=False, sep=";")

    # client_recs.to_csv("Output/Panama/rutas/D_rutas_rec.csv", mode='a', header=False, index=False,sep=";")

## 4.3 Corriendo PS

In [86]:
def runALS():
    # Ruta del archivo que deseas verificar y eliminar si existe
    ruta_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo existe
    if os.path.exists(ruta_archivo):
        # Eliminar el archivo si existe
        os.remove(ruta_archivo)
        print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
    else:
        print(f"El archivo en la ruta {ruta_archivo} no existe.")
    # CORRIENDO ALS PARA RUTAS DE MAS DE 45 SKUs
    for ruta in rutas:
        print("*" * 21)
        temp = pd.read_csv(f"Processed/rutas/D_{ruta}_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en ruta {ruta}:", sku_len)
        als_training_job(ruta, sku_len)
    # CORRIENDO ALS PARA rutas que tengan menos de 45 SKUS
    if len(low_sku_ruta) != 0:
        temp = pd.read_csv(f"Processed/rutas/D_low_ruta_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en low ruta {low_sku_ruta}:", sku_len)
        als_training_job("low_ruta", sku_len)
    else:
        print("No existen rutas con pocos skus :D")

In [87]:
rutas = rutas[~pd.Series(rutas).isin(pd.Series(low_sku_ruta))]
rutas

array([11303, 11302, 11601, 11301, 11706, 11604, 11702, 11207, 11707,
       11502, 11801, 11705, 11204, 11206, 11202, 11104, 11704, 11602,
       11504, 11703, 11802, 11708, 11203, 11701, 11102, 11503, 11205,
       11103, 11401, 11404, 11101, 11201, 11402, 11501, 11901, 11105,
       11403, 11803, 11405, 11603, 11902])

In [88]:
runALS()

El archivo en la ruta Output/D_rutas_rec.csv ha sido eliminado.
*********************
SKUs disponibles en ruta 11303: 50


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/30 21:45:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/30 21:45:56 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/30 21:45:56 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
26/03/30 21:45:56 WARN InstanceBuilder$NativeLAPACK: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


*********************
SKUs disponibles en ruta 11302: 53
*********************
SKUs disponibles en ruta 11601: 49
*********************
SKUs disponibles en ruta 11301: 50
*********************
SKUs disponibles en ruta 11706: 49
*********************
SKUs disponibles en ruta 11604: 49
*********************
SKUs disponibles en ruta 11702: 49
*********************
SKUs disponibles en ruta 11207: 50
*********************
SKUs disponibles en ruta 11707: 48
*********************
SKUs disponibles en ruta 11502: 49
*********************
SKUs disponibles en ruta 11801: 48
*********************
SKUs disponibles en ruta 11705: 48
*********************
SKUs disponibles en ruta 11204: 50
*********************
SKUs disponibles en ruta 11206: 49
*********************
SKUs disponibles en ruta 11202: 50
*********************
SKUs disponibles en ruta 11104: 48
*********************
SKUs disponibles en ruta 11704: 47
*********************
SKUs disponibles en ruta 11602: 49
*********************
SKUs disp

# 5. Ajustes al output del algoritmo ALS

## 5.-9 Quedarnos con skus que hayan tenido ventas en los ultimos 30 dias

In [89]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [90]:
pan_rec.nunique()

id_cliente            2247
cod_articulo_magic      65
dtype: int64

In [91]:
PS_VAL_promedio_recs_por_cliente_modelo = (
    pan_rec.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_modelo

49.40364931019137

In [92]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [93]:
df_ventas = pd.read_parquet("Processed/pan_ventas_manana.parquet")

In [94]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(df_ventas["fecha_liquidacion"])

In [95]:
# Definir el umbral de tiempo (últimos 30 días desde hoy)
# bajamos de 30 a 14 dias 2025-07-25
hoy = pd.Timestamp.today()
fecha_limite = hoy - pd.Timedelta(days=14)
print(fecha_limite)
# Filtrar ventas en los últimos 30 días
ventas_filtradas = df_ventas[df_ventas["fecha_liquidacion"] >= fecha_limite]

2026-03-16 21:48:08.961993


In [96]:
# Obtener los productos vendidos en cada ruta
productos_por_ruta = ventas_filtradas.groupby("cod_ruta")["cod_articulo_magic"].unique().reset_index()

In [97]:
# Unir con los productos vendidos en cada ruta
recomendaciones_validas = pan_rec.merge(productos_por_ruta, on="cod_ruta", how="inner")

In [98]:
recomendaciones_validas.columns = ['id_cliente', 'cod_articulo_magic', 'cod_ruta','lista_sku_ruta']

In [99]:
recomendaciones_validas.shape

(111010, 4)

In [100]:
recomendaciones_validas  = recomendaciones_validas[recomendaciones_validas.apply(lambda row: row["cod_articulo_magic"] in row["lista_sku_ruta"], axis=1)].reset_index(drop=True)

In [101]:
recomendaciones_validas.shape

(87195, 4)

In [102]:
recomendaciones_validas.head()

,id_cliente,cod_articulo_magic,cod_ruta,lista_sku_ruta
0,CAM|0096|28160,523619,11303,"[523619, 522026, 522500, 523371, 524091, 52387..."
1,CAM|0096|17880,524091,11303,"[523619, 522026, 522500, 523371, 524091, 52387..."
2,CAM|0096|90590,522866,11303,"[523619, 522026, 522500, 523371, 524091, 52387..."
3,CAM|0096|17450,523370,11303,"[523619, 522026, 522500, 523371, 524091, 52387..."
4,CAM|0096|85720,622061,11303,"[523619, 522026, 522500, 523371, 524091, 52387..."


In [103]:
pan_rec = recomendaciones_validas[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [104]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0096|28160,523619
1,CAM|0096|17880,524091
2,CAM|0096|90590,522866
3,CAM|0096|17450,523370
4,CAM|0096|85720,622061


In [105]:
pan_rec.shape

(87195, 2)

In [106]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-8 Recomendar de acuerdo a Subida, Bajada, Mantener

In [107]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
#pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [108]:
pan_rec.shape

(87195, 2)

In [109]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [110]:
df_ventas = pd.read_parquet("Processed/pan_ventas_manana.parquet")

In [111]:
def desplazar_nulos_fila(fila):
    return pd.Series(sorted(fila, key=pd.isna), index=fila.index)

In [112]:
# Definir una función para asignar los valores subida, mantener y bajar
def clasificar_valor(x):
    if x > 0:
        return "S"
    elif x == 0:
        return "M"
    else:
        return "B"

In [113]:
df_ventas['fecha_liquidacion'] = pd.to_datetime(df_ventas['fecha_liquidacion'])

# Definir la fecha de referencia (hoy)
fecha_actual = datetime.now(pytz.timezone("America/Lima"))#.strftime('%Y-%m-%d')
fecha_30dias_atras = (fecha_actual - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
fecha_60dias_atras = (fecha_actual - pd.Timedelta(days=60)).strftime('%Y-%m-%d')

print("fecha_actual",fecha_actual)
print("fecha_30dias_atras",fecha_30dias_atras)
print("fecha_60dias_atras",fecha_60dias_atras)

fecha_actual 2026-03-30 16:48:10.711073-05:00
fecha_30dias_atras 2026-02-28
fecha_60dias_atras 2026-01-29


In [114]:
# Filtrar los últimos 30 días
df_ultimos_30 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_30dias_atras) 
                          & (df_ventas['fecha_liquidacion'] <= fecha_actual.strftime('%Y-%m-%d'))]

# Filtrar los días 31 a 60
df_31_60 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_60dias_atras)
              & (df_ventas['fecha_liquidacion'] <= fecha_30dias_atras)]

# Agrupar por codigo_usuario, ruta, codigo_producto y sumar ventas
ventas_ultimos_30 = df_ultimos_30.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_ultimos_30["mes"] = "0_30"
ventas_31_60 = df_31_60.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_31_60["mes"] = "31_60"

df_grouped = pd.concat([ventas_ultimos_30,ventas_31_60],axis=0,ignore_index=True)
df_grouped = pd.pivot_table(
    df_grouped,
    values="imp_netovta",
    index=["cod_ruta", "cod_articulo_magic"],
    columns=["mes"],
    aggfunc="sum",
).reset_index()

# Rellenando con 0 si es nulo
df_grouped["0_30"] = df_grouped["0_30"].fillna(0)
df_grouped["31_60"] = df_grouped["31_60"].fillna(0)

# Asignando variacion de productos por mes
df_grouped["v1_2"] = (
    (df_grouped[df_grouped.columns[2]] - df_grouped[df_grouped.columns[3]])
    / df_grouped[df_grouped.columns[3]]
    * 100
)
df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)
df_grouped["v1_2"] = df_grouped["v1_2"].fillna(-1)
df_grouped["vp"] = (df_grouped["v1_2"])

/tmp/ipykernel_30399/261865676.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)


In [115]:
# Aplicar la función a la columna 'valor' y guardar el resultado en una nueva columna 'flag'
df_grouped["flag"] = df_grouped["vp"].apply(lambda x: clasificar_valor(x))
# Definir el mapeo de valores
mapeo_flag = {"S": 0, "M": 1, "B": 2}
# Aplicar el mapeo a la columna 'clasificacion' y guardar el resultado en una nueva columna 'flag_rank'
df_grouped["flag_rank"] = df_grouped["flag"].map(mapeo_flag)

In [116]:
df_grouped

mes,cod_ruta,cod_articulo_magic,0_30,31_60,v1_2,vp,flag,flag_rank
0,11101,500580,233.60,255.50,-8.571429,-8.571429,B,2
1,11101,500713,277.40,146.00,90.000000,90.000000,S,0
2,11101,500742,135.80,271.60,-50.000000,-50.000000,B,2
3,11101,500862,0.00,10.20,-100.000000,-100.000000,B,2
4,11101,501505,102.20,29.20,250.000000,250.000000,S,0
...,...,...,...,...,...,...,...,...
1886,11902,622108,26.00,13.00,100.000000,100.000000,S,0
1887,11902,622144,0.00,18.00,-100.000000,-100.000000,B,2
1888,11902,622402,32.00,20.00,60.000000,60.000000,S,0
1889,11902,623431,57.60,0.00,-1.000000,-1.000000,B,2


In [117]:
pan_rec.shape

(87195, 3)

In [118]:
pan_rec = pd.merge(pan_rec,df_grouped[["cod_ruta", "cod_articulo_magic","flag_rank"]],on=["cod_ruta", "cod_articulo_magic"],how="left")

In [119]:
pan_rec["flag_rank"] = pan_rec["flag_rank"].fillna(3)

In [120]:
# Agregar una columna con el orden original
pan_rec["original_order"] = pan_rec.index

In [121]:
# Ordenar por cod_cliente, rank_producto (ascendente) y mantener el orden original en caso de empate
pan_rec = pan_rec.sort_values(by=["id_cliente", "flag_rank", "original_order"], ascending=[True, True, True])#.drop(columns=["original_order"])

In [122]:
pan_rec.head()

,id_cliente,cod_articulo_magic,cod_ruta,flag_rank,original_order
51845,CAM|0096|10,622402,11203,0,51845
52167,CAM|0096|10,517192,11203,0,52167
52490,CAM|0096|10,500713,11203,0,52490
52703,CAM|0096|10,524091,11203,0,52703
53174,CAM|0096|10,522866,11203,0,53174


In [123]:
pan_rec.shape

(87195, 5)

In [124]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [125]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0096|10,622402
1,CAM|0096|10,517192
2,CAM|0096|10,500713
3,CAM|0096|10,524091
4,CAM|0096|10,522866


In [126]:
pan_rec.shape

(87195, 2)

In [127]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-7 Usamos el archivo de validacion para no recomendar SKUs que no se deben a las rutas

In [128]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [129]:
pan_rec.shape

(87195, 2)

In [130]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [131]:
pan_rec.isnull().sum()

id_cliente            0
cod_articulo_magic    0
cod_ruta              0
dtype: int64

In [132]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().head()

cod_ruta
11101    36
11102    38
11103    34
11104    29
11105    32
Name: cod_articulo_magic, dtype: int64

In [133]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().mean()

38.63414634146341

In [134]:
# Ruta del archivo en S3
s3_path = "s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/cam/maestro_productos_cam000"

# Leer el CSV directamente a un DataFrame
skus_val = pd.read_csv(s3_path,sep = ";")
# skus_val = pd.read_csv("maestra_productos_cam_20250217.csv",sep = ";")

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [135]:
skus_val = skus_val[skus_val.cod_compania==96]
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print("fecha_proceso: ", hoy)
# Filtrar solo las filas con la fecha de hoy
# skus_val = skus_val[skus_val['fecha_proceso'] == hoy]
skus_val["cod_compania"] = (skus_val["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
skus_val["id_cliente"] = (
    "CAM"
    + "|"
    + skus_val["cod_compania"].astype(str)
    + "|"
    + skus_val["cod_cliente"].astype(str)
)

fecha_proceso:  20260330


In [136]:
skus_val.head()

,cod_compania,desc_compania,cod_sucursal,desc_sucursal,cod_zona,desc_zona,cod_ruta,desc_ruta,cod_modulo,desc_modulo,cod_cliente,cod_articulo_magic,desc_articulo,id_cliente
98260,0096,"ECONORED DE PANAMÁ, S.A.",1,CEDI PANAMA,1001,DETALLE-CENTRO,11104,DETALLE,19119,11104,18998,522868,"CIFRUT NARANJA, MANDARINA Y LIMON PET NO RETOR...",CAM|0096|18998
98261,0096,"ECONORED DE PANAMÁ, S.A.",3,CEDI AGUADULCE,1005,DETALLE,11503,DETALLE,19608,11603,30686,622062,BIG COLA PET NO RETORNABLE 3030 ML 6 MC,CAM|0096|30686
98262,0096,"ECONORED DE PANAMÁ, S.A.",3,CEDI AGUADULCE,1006,DETALLE,11602,DETALLE,19507,11504,25302,523219,"CIFRUT NARANJA, MANDARINA Y LIMON PET NO RETOR...",CAM|0096|25302
98263,0096,"ECONORED DE PANAMÁ, S.A.",2,CEDI CHIRIQUI,1007,DETALLE,11706,DETALLE,19732,11706,5491,622062,BIG COLA PET NO RETORNABLE 3030 ML 6 MC,CAM|0096|5491
98264,0096,"ECONORED DE PANAMÁ, S.A.",1,CEDI PANAMA,1004,DETALLE--OESTE,11406,DETALLE,19434,11406,20734,502501,BIG GINGER ALE PET NO RETORNABLE 360 ml 24 pack,CAM|0096|20734


In [137]:
skus_val.shape

(208228, 14)

In [138]:
# skus_val = pd.read_csv("productos_pedido_sugerido_validacion_cam_090225.csv",sep=";")
# skus_val = skus_val[skus_val.cod_compania==70].reset_index(drop=True)

In [139]:
# Cada ruta pertenece a una compañia
skus_val.groupby("cod_ruta").cod_compania.nunique().mean()

1.0

In [140]:
pan_rec = pd.merge(pan_rec,skus_val[['cod_articulo_magic', 'id_cliente']].drop_duplicates(),left_on=["id_cliente","cod_articulo_magic"],right_on=["id_cliente","cod_articulo_magic"],how="inner")

In [141]:
pan_rec.shape

(33836, 3)

In [142]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().head()

cod_ruta
11101    36
11102    38
11103    34
11104    29
11105    32
Name: cod_articulo_magic, dtype: int64

In [143]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().mean()

38.63414634146341

In [144]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]]

In [145]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0096|10,622402
1,CAM|0096|10,517192
2,CAM|0096|10,500713
3,CAM|0096|10,524091
4,CAM|0096|10,522866


In [146]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-5 Filtrar SKUs por STOCK

In [147]:
stock = pd.read_csv("s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/cam/D_stock_pa.csv")

In [148]:
stock.drop(columns=["Fecha","Database"], inplace=True)
stock.columns=["cod_compania","cod_sucursal","cod_articulo_magic","stock_cf"]

In [149]:
stock["cod_compania"] = (stock["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
stock["cod_sucursal"] = (stock["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [150]:
fecha_hace_12_dias = (datetime.today() - timedelta(days=12)).strftime('%Y-%m-%d')
print(fecha_hace_12_dias)

2026-03-18


In [151]:
# Promedio diario de cajas fisicas por sucursal y compañia de los ultimos 12 dias
prom_diario_vta = df_ventas[df_ventas.cant_cajafisicavta > 0]
prom_diario_vta = df_ventas[df_ventas.fecha_liquidacion>=fecha_hace_12_dias].groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic","fecha_liquidacion"]).cant_cajafisicavta.sum().reset_index().groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic"]).cant_cajafisicavta.mean().reset_index()
prom_diario_vta["cod_compania"] = (prom_diario_vta["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
prom_diario_vta["cod_sucursal"] = (prom_diario_vta["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [152]:
df_stock = pd.merge(prom_diario_vta,stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"],how="left")
df_stock["dias_stock"] = df_stock["stock_cf"]/df_stock["cant_cajafisicavta"]
df_stock =  df_stock[(df_stock.dias_stock > 3) & (df_stock.cant_cajafisicavta > 0)].reset_index(drop=True)

In [153]:
pan_rec.nunique()

id_cliente            2247
cod_articulo_magic      59
dtype: int64

In [154]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

15.058299955496217

In [155]:
pan_rec.shape

(33836, 2)

In [156]:
# Cruzamos stock

In [157]:
test = pd.merge(pan_rec,df_ventas[["id_cliente","cod_compania","cod_sucursal"]].drop_duplicates(),on="id_cliente",how="left")

In [158]:
test["cod_compania"] = (test["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
test["cod_sucursal"] = (test["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [159]:
test_f = pd.merge(test, df_stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"], how="inner")

In [160]:
test_f.head()

,id_cliente,cod_articulo_magic,cod_compania,cod_sucursal,cant_cajafisicavta,stock_cf,dias_stock
0,CAM|0096|10,622402,0096,01,99.500000,6035,60.653266
1,CAM|0096|10,517192,0096,01,91.111111,1045,11.469512
2,CAM|0096|10,500713,0096,01,60.888889,3051,50.107664
3,CAM|0096|10,524091,0096,01,168.300000,5820,34.581105
4,CAM|0096|10,522026,0096,01,133.750000,2091,15.633645


In [161]:
test_f.shape

(31512, 7)

In [162]:
test_f.nunique()

id_cliente            2247
cod_articulo_magic      59
cod_compania             1
cod_sucursal             5
cant_cajafisicavta     156
stock_cf               208
dias_stock             230
dtype: int64

In [163]:
PS_VAL_promedio_recs_por_cliente_stock = (
    test_f.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_stock

13.918558077436582

In [164]:
pan_rec = test_f[["id_cliente","cod_articulo_magic"]].copy()

In [165]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-4 Mantener Solo SKUS en especifico

In [166]:
# pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# # pan_rec["id_cliente"]="CAM|"+pan_rec["id_cliente"]

In [167]:
# sku_con_precio = pd.read_excel(
#     "Input/LISTA DE PRECIOS PANAMA_2024-10-28.xlsx", sheet_name="activos"
# )

In [168]:
# master_prod=pd.read_csv("D_pan_masterProd.csv")

In [169]:
# master_prod[master_prod.cod_articulo_magic==522866]

In [170]:
# sku_con_precio[sku_con_precio["Producto: SKU"]==522866]

In [171]:
# ventas=pd.read_csv("Processed/ventas_panama_12m.csv")

In [172]:
# ventas[ventas.cod_articulo_magic==522866][["fecha_liquidacion","cod_sucursal","cod_articulo_magic","desc_marca","desc_articulo"]].drop_duplicates().sort_values("fecha_liquidacion")

In [173]:
# # Original size de recomendacioness
# pan_rec.shape

In [174]:
# pan_rec = pan_rec[
#     pan_rec.cod_articulo_magic.isin(sku_con_precio["Producto: SKU"].unique())
# ].reset_index(drop=True)
# pan_rec

In [175]:
# # Size de recomendaciones luego de filtar solo a los del excel de SKUs activos
# pan_rec.shape

In [176]:
# PS_VAL_total_skus_recomendados_stock_y_activos = pan_rec["cod_articulo_magic"].nunique()
# PS_VAL_total_skus_recomendados_stock_y_activos

In [177]:
# pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-3 Quitar Recomendaciones de SKUS en especifico

In [178]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [179]:
skus_sin_precio = [622402]

In [180]:
pan_rec = pan_rec[~(pan_rec["cod_articulo_magic"].isin(skus_sin_precio))].reset_index(
    drop=True
)

In [181]:
PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda = (
    pan_rec.cod_articulo_magic.nunique()
)
PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda

58

In [182]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-2 Quitar Recomendaciones de los ultimos 14 dias

In [183]:
import os
import re

In [184]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"]="CAM|"+pan_rec["id_cliente"]

In [185]:
pan_rec

,id_cliente,cod_articulo_magic
0,CAM|0096|10,517192
1,CAM|0096|10,500713
2,CAM|0096|10,524091
3,CAM|0096|10,522026
4,CAM|0096|10,523370
...,...,...
30780,CAM|0096|93497,523136
30781,CAM|0096|93498,523136
30782,CAM|0096|93498,502502
30783,CAM|0096|93498,599845


In [186]:
master_prod = pd.read_csv("Input/D_pan_masterProd.csv")

In [187]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-03-31'

In [188]:
ruta_archivo = f"Output/PS_todo_panama/D_base_pedidos_{fecha_tomorrow}.csv"
# Verificar si el archivo existe
if os.path.exists(ruta_archivo):
    # Eliminar el archivo si existe
    os.remove(ruta_archivo)
    print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
else:
    print(f"El archivo en la ruta {ruta_archivo} no existe.")

El archivo en la ruta Output/PS_todo_panama/D_base_pedidos_2026-03-31.csv no existe.


In [189]:
# Directorio donde se encuentran los archivos
directorio = "Output/PS_todo_panama/"

# Expresión regular para encontrar fechas en el nombre de los archivos
patron_fecha = r"\d{4}-\d{2}-\d{2}"

# Lista para almacenar las fechas extraídas
fechas = []

# Iterar sobre los archivos en el directorio
for archivo in os.listdir(directorio):
    # Verificar si el archivo es un archivo CSV y coincide con el patrón de nombres
    if archivo.endswith(".csv") and re.match(
        r"^D_base_pedidos_\d{4}-\d{2}-\d{2}\.csv$", archivo
    ):
        # Extraer la fecha del nombre del archivo
        fecha = re.search(patron_fecha, archivo).group()
        # Agregar la fecha a la lista
        fechas.append(fecha)

In [190]:
last_7_days_2 = sorted(fechas)[-14:]
# last_7_days_2=last_7_days_2[:-1]
last_7_days_2

['2026-03-14',
 '2026-03-16',
 '2026-03-17',
 '2026-03-18',
 '2026-03-19',
 '2026-03-20',
 '2026-03-21',
 '2026-03-23',
 '2026-03-24',
 '2026-03-25',
 '2026-03-26',
 '2026-03-27',
 '2026-03-28',
 '2026-03-30']

In [191]:
# Leer recomendaciones de los ultimos 7 dias y cruzar con clientes que tienen recomendacion para mañana
last_7_days_recs = pd.DataFrame()
for fecha_rec in last_7_days_2:
    df_temp = pd.read_csv(
        f"Output/PS_todo_panama/D_base_pedidos_{fecha_rec}.csv",
        dtype={"Compania": "str", "Cliente": "str"},
    )
    df_temp["id_cliente"] = "CAM|" + df_temp["Compania"] + "|" + df_temp["Cliente"]
    df_temp = df_temp[df_temp["id_cliente"].isin(pan_rec["id_cliente"].unique())]
    last_7_days_recs = pd.concat([last_7_days_recs, df_temp], axis=0)
    print(f"{fecha_rec} done")

2026-03-14 done
2026-03-16 done
2026-03-17 done
2026-03-18 done
2026-03-19 done
2026-03-20 done
2026-03-21 done
2026-03-23 done
2026-03-24 done
2026-03-25 done
2026-03-26 done
2026-03-27 done
2026-03-28 done
2026-03-30 done


In [192]:
# Utilizamos merge para combinar los dataframes, utilizando la columna 'Cliente' como clave
df_combinado = pd.merge(
    pan_rec,
    last_7_days_recs,
    left_on=["id_cliente", "cod_articulo_magic"],
    right_on=["id_cliente", "Producto"],
    how="left",
    indicator=True,
)
# Filtramos los registros que solo están en el DataFrame 1
df_resultado = df_combinado[df_combinado["_merge"] == "left_only"][
    ["id_cliente", "cod_articulo_magic"]
].reset_index(drop=True)
df_resultado

,id_cliente,cod_articulo_magic
0,CAM|0096|10,517192
1,CAM|0096|10,524091
2,CAM|0096|10,523370
3,CAM|0096|10,599844
4,CAM|0096|10,623432
...,...,...
22956,CAM|0096|93497,523136
22957,CAM|0096|93498,523136
22958,CAM|0096|93498,502502
22959,CAM|0096|93498,599845


In [193]:
pan_rec.shape

(30785, 2)

In [194]:
pan_rec.id_cliente.nunique()

2246

In [195]:
pan_rec.cod_articulo_magic.nunique()

58

In [196]:
df_resultado.shape

(22961, 2)

In [197]:
df_resultado.id_cliente.nunique()

2205

In [198]:
df_resultado.cod_articulo_magic.nunique()

58

In [199]:
df_resultado

,id_cliente,cod_articulo_magic
0,CAM|0096|10,517192
1,CAM|0096|10,524091
2,CAM|0096|10,523370
3,CAM|0096|10,599844
4,CAM|0096|10,623432
...,...,...
22956,CAM|0096|93497,523136
22957,CAM|0096|93498,523136
22958,CAM|0096|93498,502502
22959,CAM|0096|93498,599845


In [200]:
df_resultado.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.0 Calcular Irregularidad de clientes

In [201]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [202]:
pan_rec["id_cliente"].nunique()

2205

In [203]:
now = pd.to_datetime(datetime.now(pytz.timezone("America/Lima")).strftime("%Y-%m-01"))
fecha_doce_meses_atras = now - pd.DateOffset(months=12)
lista_m12 = [fecha_doce_meses_atras + pd.DateOffset(months=i) for i in range(12)]
lista_m12

fecha_seis_meses_atras = now - pd.DateOffset(months=6)
lista_m6 = [fecha_seis_meses_atras + pd.DateOffset(months=i) for i in range(6)]
lista_m6

[Timestamp('2025-09-01 00:00:00'),
 Timestamp('2025-10-01 00:00:00'),
 Timestamp('2025-11-01 00:00:00'),
 Timestamp('2025-12-01 00:00:00'),
 Timestamp('2026-01-01 00:00:00'),
 Timestamp('2026-02-01 00:00:00')]

In [204]:
temp_df_ventas = df_ventas[df_ventas["id_cliente"].isin(pan_rec["id_cliente"].unique())]

In [205]:
qw = (
    temp_df_ventas[["id_cliente", "mes"]]
    .drop_duplicates()
    .sort_values(["id_cliente", "mes"])
    .groupby("id_cliente")
    .tail(12)
    .reset_index(drop=True)
)
qw["mes"] = pd.to_datetime(qw["mes"])
qw["m12"] = qw["mes"].isin(lista_m12)
qw["m6"] = qw["mes"].isin(lista_m6)

In [206]:
categoria_cliente = qw.groupby("id_cliente")[["m12", "m6"]].sum().reset_index()

In [207]:
# Definir condiciones
condicion_super = categoria_cliente["m12"] == 12
condicion_frecuente = (
    (categoria_cliente["m12"] < 12)
    & (categoria_cliente["m12"] >= 6)
    & (categoria_cliente["m6"] == 6)
)
condicion_regulares = (categoria_cliente["m6"] <= 6) & (categoria_cliente["m12"] <= 6)
condicion_riesgo = (categoria_cliente["m6"] < 6) & (categoria_cliente["m6"] >= 4)
condicion_irregulares = categoria_cliente["m6"] < 4

# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_super, "categoria_cliente"] = "Super"
categoria_cliente.loc[condicion_frecuente, "categoria_cliente"] = "Frecuente"
categoria_cliente.loc[condicion_regulares, "categoria_cliente"] = "Regular"
categoria_cliente.loc[condicion_riesgo, "categoria_cliente"] = "Riesgo"
categoria_cliente.loc[condicion_irregulares, "categoria_cliente"] = "Irregular"

**Recalcular Irregularidad**

In [208]:
# Definir condiciones
condicion_regular = (
    (categoria_cliente["categoria_cliente"] == "Super")
    | (categoria_cliente["categoria_cliente"] == "Frecuente")
    | (categoria_cliente["categoria_cliente"] == "Regular")
    | (categoria_cliente["categoria_cliente"] == "Riesgo")
)
condicion_irregular = categoria_cliente["categoria_cliente"] == "Irregular"
# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_regular, "categoria_cliente_2"] = "Regular"
categoria_cliente.loc[condicion_irregular, "categoria_cliente_2"] = "Irregular"

In [209]:
categoria_cliente

,id_cliente,m12,m6,categoria_cliente,categoria_cliente_2
0,CAM|0096|10,11,6,Frecuente,Regular
1,CAM|0096|10030,11,6,Frecuente,Regular
2,CAM|0096|10031,11,6,Frecuente,Regular
3,CAM|0096|10035,11,6,Frecuente,Regular
4,CAM|0096|10042,11,6,Frecuente,Regular
...,...,...,...,...,...
2200,CAM|0096|93482,0,0,Irregular,Irregular
2201,CAM|0096|93483,0,0,Irregular,Irregular
2202,CAM|0096|93484,0,0,Irregular,Irregular
2203,CAM|0096|93497,0,0,Irregular,Irregular


In [210]:
categoria_cliente["categoria_cliente"].value_counts(dropna=False)

categoria_cliente
Frecuente    1416
Irregular     380
Riesgo        374
Regular        22
Super          13
Name: count, dtype: int64

In [211]:
categoria_cliente["categoria_cliente_2"].value_counts(dropna=False)

categoria_cliente_2
Regular      1825
Irregular     380
Name: count, dtype: int64

## 5.1. Obteniendo marcas de recomendaciones por cliente

In [212]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [213]:
pan_ventas = df_ventas[(df_ventas["cod_ruta"].isin(rutas))]
pan_ventas2 = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
pan_ventas = pd.concat([pan_ventas, pan_ventas2], axis=0)

In [214]:
pan_evo_venta = pd.read_csv("Input/D_pan_EvolucionCompra.csv")

In [215]:
%%time


def desplazar_nulos_fila(fila):
    return pd.Series(sorted(fila, key=pd.isna), index=fila.index)


# Convertir las Ventas de Nuevos Productos a 0 para que luego no sea afectada al desplazar los nulos
for i in range(int((pan_evo_venta.shape[1] - 4) / 5)):
    pan_evo_venta.loc[
        (
            (~pan_evo_venta[f"mes_{i+1}_ventaBeb"].isnull())
            & (pan_evo_venta[f"mes_{i+1}_ventaNP"].isnull())
        ),
        f"mes_{i+1}_ventaNP",
    ] = 0
    pan_evo_venta.loc[
        (
            (pan_evo_venta[f"mes_{i+1}_ventaBeb"].isnull())
            & (~pan_evo_venta[f"mes_{i+1}_ventaNP"].isnull())
        ),
        f"mes_{i+1}_ventaBeb",
    ] = 0

# Desplazar los meses nulos a la derecha
pan_evo_venta = pan_evo_venta.apply(desplazar_nulos_fila, axis=1)
pan_evo_venta

CPU times: user 2.16 s, sys: 76 ms, total: 2.23 s
Wall time: 2.21 s


,id_cliente,fecha_creacion,tipo_cliente,len_m,continuo,venta,venta_bebidas,venta_newprod,mes_1_marca,mes_1_marcaCount,...,mes_45_marca,mes_45_marcaCount,mes_45_venta,mes_45_ventaBeb,mes_45_ventaNP,mes_46_marca,mes_46_marcaCount,mes_46_venta,mes_46_ventaBeb,mes_46_ventaNP
0,CAM|0096|10,2009-07-27,OLD,46,True,34914.2006,34844.65,69.5506,"BIG, CIFRUT",2,...,"BIG, CIFRUT, D GUSSTO",3.0,723.15,700.35,22.8,"BIG, CIFRUT, SPORADE",3.0,503.00,503.00,0.0
1,CAM|0096|1002,2009-07-29,OLD,46,True,44621.4000,44562.60,58.8000,"BIG, CIFRUT",2,...,"BIG, CIFRUT, D GUSSTO, KR",4.0,1034.20,1028.50,5.7,"BIG, BIO ALOE, CIFRUT, KR, SPORADE",5.0,1692.70,1692.70,0.0
2,CAM|0096|10027,2014-10-21,OLD,22,False,2900.8200,2900.82,0.0000,CIFRUT,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CAM|0096|10028,2014-10-21,OLD,46,True,16640.1300,16634.43,5.7000,"BIG, CIFRUT",2,...,"BIG, CIFRUT",2.0,639.70,639.70,0.0,"BIG, CIFRUT",2.0,555.20,555.20,0.0
4,CAM|0096|10029,2014-10-21,OLD,46,True,8672.5006,8639.25,33.2506,"BIG, CIFRUT",2,...,"BIG, CIFRUT",2.0,74.00,74.00,0.0,"BIG, CIFRUT",2.0,65.85,65.85,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9095,CAM|0096|995,2009-07-29,OLD,46,True,14153.2400,14131.94,21.3000,CIFRUT,1,...,"BIG, CIELO, CIFRUT",3.0,442.10,442.10,0.0,"BIG, CIFRUT",2.0,451.35,451.35,0.0
9096,CAM|0096|9957,2014-10-01,OLD,7,True,522.9000,522.90,0.0000,"BIG, CIELO, CIFRUT, SPORADE",4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9097,CAM|0096|9961,2014-10-06,OLD,46,True,1696.2000,1696.20,0.0000,CIFRUT,1,...,CIFRUT,1.0,52.70,52.70,0.0,CIFRUT,1.0,35.75,35.75,0.0
9098,CAM|0096|997,2009-07-29,OLD,46,True,28593.6800,28593.68,0.0000,"BIG, CIFRUT",2,...,"BIG, CIFRUT",2.0,574.35,574.35,0.0,"BIG, CIFRUT",2.0,320.10,320.10,0.0


In [216]:
marca_articulo = pan_ventas[["desc_categoria", "cod_articulo_magic"]].drop_duplicates()
print(marca_articulo.shape)
marca_articulo.head()

(65, 2)


,desc_categoria,cod_articulo_magic
0,AGUA,508426
1,JUGOS LIGEROS,509671
2,GASEOSAS,500862
3,JUGOS LIGEROS,523136
4,AGUA,522026


In [217]:
cliente_rec_marca = pd.merge(
    pan_rec, marca_articulo, on="cod_articulo_magic", how="left"
)
cliente_rec_marca["desc_categoria"] = cliente_rec_marca["desc_categoria"].str.strip()
print(cliente_rec_marca.shape)
cliente_rec_marca

(22961, 3)


,id_cliente,cod_articulo_magic,desc_categoria
0,CAM|0096|10,517192,ENERGIZANTE
1,CAM|0096|10,524091,AGUA
2,CAM|0096|10,523370,SUERO
3,CAM|0096|10,599844,GASEOSAS
4,CAM|0096|10,623432,SOPAS
...,...,...,...
22956,CAM|0096|93497,523136,JUGOS LIGEROS
22957,CAM|0096|93498,523136,JUGOS LIGEROS
22958,CAM|0096|93498,502502,GASEOSAS
22959,CAM|0096|93498,599845,GASEOSAS


## 5.2. Calcular marcas distintas recomendadas por cliente

In [218]:
# Conteo de categorias recomendadas por Cliente
cods2 = cliente_rec_marca.groupby("id_cliente")["desc_categoria"].nunique().reset_index()
print(cods2.shape)
cods2.head()

(2205, 2)


,id_cliente,desc_categoria
0,CAM|0096|10,8
1,CAM|0096|10030,5
2,CAM|0096|10031,7
3,CAM|0096|10035,7
4,CAM|0096|10042,4


## 5.3 Quitando los SKUs de las ultimas 2 semanas (Evitar recompra)

In [219]:
last_2_weeks = pd.to_datetime(datetime.now(pytz.timezone("America/Lima"))) - pd.DateOffset(
    days=14
)
last_2_weeks

Timestamp('2026-03-16 16:48:34.312922-0500', tz='America/Lima')

In [220]:
# Asegurarse de que 'fecha_liquidacion' esté en formato datetime
pan_ventas["fecha_liquidacion"] = pd.to_datetime(pan_ventas["fecha_liquidacion"])
# Convertir 'fecha_liquidacion' a la misma zona horaria que 'last_year'
pan_ventas["fecha_liquidacion"] = pan_ventas["fecha_liquidacion"].dt.tz_localize(
    "America/Lima", nonexistent="NaT", ambiguous="NaT"
)

In [221]:
df_quitar_2_weeks = pan_ventas[pan_ventas["fecha_liquidacion"]>=last_2_weeks][["id_cliente","cod_articulo_magic"]].drop_duplicates().reset_index(drop=True)

In [222]:
# Hacemos un merge para identificar coincidencias
cliente_rec_sin4 = cliente_rec_marca.merge(df_quitar_2_weeks, on=['id_cliente', 'cod_articulo_magic'], how='left', indicator=True)
# Filtramos solo los registros que NO están en df_quitar_2_weeks
cliente_rec_sin4 = cliente_rec_sin4[cliente_rec_sin4['_merge'] == 'left_only'].drop(columns=['_merge'])

In [223]:
cliente_rec_marca.id_cliente.nunique()

2205

In [224]:
cliente_rec_sin4.id_cliente.nunique()

2129

In [225]:
cliente_rec_sin4

,id_cliente,cod_articulo_magic,desc_categoria
0,CAM|0096|10,517192,ENERGIZANTE
1,CAM|0096|10,524091,AGUA
3,CAM|0096|10,599844,GASEOSAS
4,CAM|0096|10,623432,SOPAS
5,CAM|0096|10,501109,GASEOSAS
...,...,...,...
22917,CAM|0096|93285,622061,GASEOSAS
22918,CAM|0096|93331,523136,JUGOS LIGEROS
22919,CAM|0096|93331,501505,GASEOSAS
22920,CAM|0096|93331,522026,AGUA


## 5.4 Filtro para calcula antiguedad de clientes

In [226]:
pan_ventas = pan_ventas.rename(columns={"fecha_creacion_cliente": "fecha_creacion"})

In [227]:
pan_ventas.dtypes

id_cliente                                    object
id_sucursal                                   object
id_producto                                   object
fecha_liquidacion       datetime64[ns, America/Lima]
cod_zona                                       int64
cant_cajafisicavta                           float64
cant_cajaunitvta                             float64
imp_netovta                                  float64
cod_compania                                  object
desc_compania                                 object
cod_sucursal                                   int64
desc_sucursal                                 object
cod_pais                                      object
fecha_creacion                               float64
cod_cliente                                   object
desc_marca                                    object
desc_formato                                 float64
desc_categoria                                object
cod_giro                                     f

In [228]:
pan_ventas["fecha_creacion"] = pan_ventas["fecha_creacion"].astype(int)

In [229]:
# Convertir la columna 'fecha_creacion' a tipo fecha
pan_ventas["fecha_creacion"] = pd.to_datetime(
    pan_ventas["fecha_creacion"], format="%Y%m%d", errors="coerce"
)

In [230]:
# Definir una función para etiquetar los clientes según su fecha de creación
def etiquetar_cliente(fecha_creacion):
    if pd.isnull(fecha_creacion):
        return (
            "nf"  # Si la fecha de creación es nula, etiquetar como 'nf' (no encontrada)
        )
    else:
        hoy = datetime.now()  # Obtener la fecha actual
        hace_12_meses = hoy - timedelta(days=365)  # Calcular hace 12 meses
        if fecha_creacion >= hace_12_meses:
            return "new_client"  # Si la fecha de creación está dentro de los últimos 12 meses, etiquetar como 'new_client'
        else:
            return "old_client"  # Si la fecha de creación es anterior a los últimos 12 meses, etiquetar como 'old_client'

In [231]:
# Aplicar la función a la columna 'fecha_creacion' para crear la nueva columna 'tipo_cliente'
pan_ventas["antiguedad"] = pan_ventas["fecha_creacion"].apply(etiquetar_cliente)

In [232]:
pan_antiguedad_clientes = (
    pan_ventas[["id_cliente", "fecha_creacion", "antiguedad"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

## 5.4.2 Obteniendo datos necesarios para armar DF para subir a SalesForce

In [233]:
# pan_ventas.groupby(["id_cliente","cod_compania","cod_sucursal","cod_cliente","cod_modulo"])[["cant_cajafisicavta","cant_cajaunitvta"]].sum().reset_index()

In [234]:
datos_para_salesforce = (
    pan_ventas.groupby(
        ["id_cliente", "cod_compania", "cod_sucursal", "cod_cliente", "cod_ruta"]
    )[
        [
            "id_cliente",
            "cod_compania",
            "cod_sucursal",
            "cod_cliente",
            "cod_modulo",
            "cod_ruta",
        ]
    ]
    .head(1)
    .reset_index(drop=True)
)

## 5.5 Juntando las Recomendaciones Finales
 - Ordenado de acuerdo al peso de la recomendación y el id de usuario.
 - Hay 5 skus de 5 marcas distintas que se recomiendan a cada usuario
 - Ningún SKU que haya comprado el usuario en sus ultimas X visitas será recomendado. La cantidad de visitas (X) a ignorar depende de sus visitas promedio mensuales.

In [235]:
# Obtener primer SKU de cada marca de la recomendacion
final_rec = cliente_rec_sin4.groupby(["id_cliente", "desc_categoria"]).first().reset_index()
# Agregar Marca de sus primeras compras
final_rec = pd.merge(final_rec, cods2, how="left", on="id_cliente")
# Agregar descripcion de SKU recomendado
pan_prod = pan_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
pan_prod = pan_prod.groupby(["cod_articulo_magic"]).first().reset_index()
final_rec = pd.merge(final_rec, pan_prod, how="left", on="cod_articulo_magic")
# Cambiar nombre de columnas
final_rec.columns = ["id_cliente", "marca_rec", "sku", "len_marca_rec", "desc_rec"]
# Agregar Giro del cliente
giros = df_ventas[df_ventas["id_cliente"].isin(final_rec["id_cliente"].unique())][
    ["id_cliente", "desc_giro", "desc_subgiro"]
].drop_duplicates()
final_rec = pd.merge(final_rec, giros, on="id_cliente", how="left")
# Agregar si el cliente es Regular o Irregular
final_rec = pd.merge(
    final_rec,
    categoria_cliente[["id_cliente", "categoria_cliente_2"]],
    on="id_cliente",
    how="left",
)
final_rec.loc[final_rec["categoria_cliente_2"] == "Irregular", "desc_giro"] = np.nan
# Agregar antiguedad de cliente
final_rec = pd.merge(final_rec, pan_antiguedad_clientes, how="left", on="id_cliente")
# Agregar segmento de cliente
info_segmentos = pan_ventas[
    [
        "id_cliente",
        "new_segment",
        "dias_de_visita__c",
        "periodo_de_visita__c",
        "ultima_visita",
    ]
].drop_duplicates()
info_segmentos = info_segmentos.groupby(["id_cliente"]).first().reset_index()
final_rec = pd.merge(final_rec, info_segmentos, how="left", on="id_cliente")
# Agregar datos necesarios para SALESFORCE
final_rec = pd.merge(final_rec, datos_para_salesforce, how="left", on="id_cliente")
# Agregar Peso de marcas dependiendo del giro del cliente
final_rec["peso"] = final_rec.apply(
    lambda row: mapeo_diccionario.get(row["desc_subgiro"], {}).get(row["marca_rec"], 5),
    axis=1,
)
# Ordenar la recomendacion de acuerdo al peso de la marca y nos quedamos con las 5 primeras marcas
final_rec = (
    final_rec.sort_values(["id_cliente", "peso"]).groupby(["id_cliente"]).head(5)
)
# Agregamos rank de la marca recomendada
final_rec["marca_rec_rank"] = final_rec.groupby("id_cliente").cumcount() + 1
final_rec = final_rec.reset_index(drop=True)

In [236]:
final_rec["sku"] = final_rec["sku"].astype(int)
final_rec["peso"] = final_rec["peso"].astype(int)
final_rec["marca_rec_rank"] = final_rec["marca_rec_rank"].astype(int)

## 5.8. Guardando recomendacion para D&A y Comercial

In [237]:
final_rec.id_cliente.nunique()

2129

In [238]:
final_rec.dias_de_visita__c.value_counts(dropna=False)

dias_de_visita__c
2      5090
2;5    1252
Name: count, dtype: int64

In [239]:
final_rec.periodo_de_visita__c.value_counts(dropna=False)

periodo_de_visita__c
F1    6342
Name: count, dtype: int64

### 5.8.1 Para D&A

In [240]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-03-31'

In [241]:
PS_VAL_total_clientes_con_visita_no_venta = final_rec.id_cliente.nunique()
PS_VAL_total_clientes_con_visita_no_venta

2129

In [242]:
final_rec["cod_cliente"] = final_rec["id_cliente"].str.split("|").str[-1]

In [243]:
PS_VAL_marcas_recomendadas = final_rec.marca_rec.unique()
PS_VAL_marcas_recomendadas

array(['GASEOSAS', 'JUGOS LIGEROS', 'AGUA', 'ENERGIZANTE',
       'BEBIDAS DEPORTIVAS', 'SUERO', 'DETERGENTE', 'SOPAS', 'PASTAS'],
      dtype=object)

In [244]:
final_rec.groupby("dias_de_visita__c")["id_cliente"].nunique()

dias_de_visita__c
2      1748
2;5     381
Name: id_cliente, dtype: int64

In [245]:
final_rec.to_csv(
    f"Output/PS_todo_panama_data/D_pan_recs_data_{fecha_tomorrow}.csv", index=False
)

In [246]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Panama/Output/PS_todo_panama_data/D_pan_recs_data_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(final_rec, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Panama/Output/PS_todo_panama_data/D_pan_recs_data_2026-03-31.csv'],
 'partitions_values': {}}

### PRIORIZAR SKUs

In [247]:
# skus_a_priorizar = [523370, 523371, 523374, 523620, 523619, 517192]

In [248]:
# skus_a_priorizar

In [249]:
# skus_a_priorizar_p1 = pan_ventas[pan_ventas.cod_articulo_magic.isin(skus_a_priorizar)].groupby(
#     ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "cod_articulo_magic"]).cant_cajafisicavta.sum().reset_index().sort_values(
#     ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "cant_cajafisicavta"], ascending = [True,True,True,True,False]).reset_index(drop=True).groupby(
#     ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo"]).head(1).reset_index(drop=True)

In [250]:
# skus_a_priorizar_p1 = skus_a_priorizar_p1.drop('cant_cajafisicavta', axis=1)

In [251]:
# cli_faltantes = final_rec[~final_rec.cod_cliente.isin(skus_a_priorizar_p1.cod_cliente.astype(str).unique())].reset_index(drop=True)

In [252]:
# skus_a_priorizar_p2 = pan_ventas[pan_ventas.id_cliente.isin(cli_faltantes.id_cliente.unique())][["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo"]].drop_duplicates().reset_index(drop=True)

In [253]:
# # Para asignar un valor aleatorio a cada fila:
# skus_a_priorizar_p2['cod_articulo_magic'] = np.random.choice(skus_a_priorizar, size=len(skus_a_priorizar_p2))

In [254]:
# skus_a_priorizar_final = pd.concat([skus_a_priorizar_p1, skus_a_priorizar_p2], axis=0, ignore_index=True)

In [255]:
# skus_a_priorizar_final.rename(columns={'cod_articulo_magic': 'sku'}, inplace=True)

In [256]:
# skus_a_priorizar_p1.cod_cliente.nunique()

In [257]:
# skus_a_priorizar_p2.cod_cliente.nunique()

In [258]:
# skus_a_priorizar_final.cod_cliente.nunique()

In [259]:
# final_rec = pd.concat([skus_a_priorizar_final, final_rec], axis=0, ignore_index=True)

## 5.9 Filtro de Recomendaciones por Segmento de cliente

In [260]:
rutas_especiales = [11103, 11205, 11301, 11401]
# reglas de cortes para rutas especiales
reglas_rutas_especiales = {
    'BLINDAR': 2,
    'MANTENER': 2,
    'DESARROLLAR': 3,
    'OPTIMIZAR': 2
}
# Definimos las reglas de cortes para las demás rutas
reglas_otras_rutas = {
    'BLINDAR': 1,
    'MANTENER': 2,
    'DESARROLLAR': 3,
    'OPTIMIZAR': 4
}

In [261]:
# aplicar la lógica de head según ruta y segmento
def aplicar_head(grupo):
    cod_ruta = grupo['cod_ruta'].iloc[0]
    segmento = grupo['new_segment'].iloc[0]

    # Seleccionar reglas dependiendo de si es ruta especial o no
    if cod_ruta in rutas_especiales:
        limite = reglas_rutas_especiales.get(segmento, len(grupo))
    else:
        limite = reglas_otras_rutas.get(segmento, len(grupo))

    return grupo.head(limite)

In [262]:
# groupby sin perder el orden original
final_rec = (
    final_rec.groupby(['cod_ruta', 'new_segment', 'id_cliente'], group_keys=False)
      .apply(aplicar_head)
)

/tmp/ipykernel_30399/4176210446.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(aplicar_head)


In [263]:
final_rec.groupby(["new_segment","id_cliente"]).size().groupby("new_segment").mean()

new_segment
BLINDAR        1.133621
DESARROLLAR    2.588946
MANTENER       1.850716
OPTIMIZAR      2.155609
dtype: float64

In [264]:
final_rec[final_rec.cod_ruta.isin(rutas_especiales)].groupby(["new_segment","id_cliente"]).size().groupby("new_segment").mean()

new_segment
BLINDAR        1.939394
DESARROLLAR    2.623377
MANTENER       1.774194
OPTIMIZAR      1.698630
dtype: float64

In [265]:
final_rec.shape

(4454, 21)

In [266]:
# # Definir una función para filtrar las filas según el valor de 'new_segment'
# def filtrar_segmento(group):
#     segmento = group["new_segment"].iloc[
#         0
#     ]  # Obtener el valor del segmento para el grupo
#     if segmento == "BLINDAR":
#         return group.head(1)  # Si el segmento es 'Blindar', mantener la primera fila
#     elif segmento == "MANTENER":
#         return group.head(
#             2
#         )  # Si el segmento es 'Mantener', mantener las dos primeras filas
#     elif segmento == "DESARROLLAR":
#         return group.head(
#             3
#         )  # Si el segmento es 'Desarrollar', mantener las tres primeras filas
#     elif segmento == "OPTIMIZAR":
#         return group.head(
#             4
#         )  # Si el segmento es 'Optimizar', mantener las cuatro primeras filas
#     else:
#         return group  # Si el segmento no es ninguno de los especificados, mantener todas las filas

In [267]:
PS_VAL_promedio_recs_por_cliente_negocio = (
    final_rec.groupby("id_cliente")["sku"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_negocio

2.0789102865194926

In [268]:
# final_rec = (
#     final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)
# )

In [269]:
final_rec["cod_compania"] = final_rec["cod_compania"].astype(int)
final_rec["cod_sucursal"] = final_rec["cod_sucursal"].astype(int)
final_rec["cod_cliente"] = final_rec["cod_cliente"].astype(int)
final_rec["cod_modulo"] = final_rec["cod_modulo"].astype(int)
final_rec["sku"] = final_rec["sku"].astype(int)

## 5.10. Dar formato para subir a SF

In [270]:
recomendaciones_para_salesforce = final_rec[
    ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "sku"]
]
recomendaciones_para_salesforce = recomendaciones_para_salesforce.drop_duplicates()
recomendaciones_para_salesforce["Pais"] = "PA"
recomendaciones_para_salesforce["Cajas"] = int(1)
recomendaciones_para_salesforce["Unidades"] = int(0)
recomendaciones_para_salesforce["Fecha"] = fecha_tomorrow
recomendaciones_para_salesforce = recomendaciones_para_salesforce[
    [
        "Pais",
        "cod_compania",
        "cod_sucursal",
        "cod_cliente",
        "cod_modulo",
        "sku",
        "Cajas",
        "Unidades",
        "Fecha",
    ]
]
recomendaciones_para_salesforce.columns = [
    "Pais",
    "Compania",
    "Sucursal",
    "Cliente",
    "Modulo",
    "Producto",
    "Cajas",
    "Unidades",
    "Fecha",
]

In [271]:
recomendaciones_para_salesforce["Compania"] = recomendaciones_para_salesforce[
    "Compania"
].apply(lambda x: str(int(x)).rjust(4, "0"))
recomendaciones_para_salesforce["Sucursal"] = recomendaciones_para_salesforce[
    "Sucursal"
].apply(lambda x: str(int(x)).rjust(2, "0"))

In [272]:
recomendaciones_para_salesforce.dtypes

Pais        object
Compania    object
Sucursal    object
Cliente      int64
Modulo       int64
Producto     int64
Cajas        int64
Unidades     int64
Fecha       object
dtype: object

In [273]:
rutas_cliente = df_ventas[["cod_cliente", "cod_ruta"]].drop_duplicates()

In [274]:
recomendaciones_para_salesforce.Sucursal.unique()

array(['01', '03', '02', '04', '05'], dtype=object)

In [275]:
recomendaciones_para_salesforce.Compania.unique()

array(['0096'], dtype=object)

In [276]:
recomendaciones_para_salesforce.Cliente.nunique()

2129

In [277]:
recomendaciones_para_salesforce

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha
597,PA,0096,01,14344,19102,500713,1,0,2026-03-31
608,PA,0096,01,14355,19102,500742,1,0,2026-03-31
613,PA,0096,01,14356,19102,502501,1,0,2026-03-31
614,PA,0096,01,14356,19102,509671,1,0,2026-03-31
615,PA,0096,01,14356,19102,522450,1,0,2026-03-31
...,...,...,...,...,...,...,...,...,...
5844,PA,0096,05,90699,19907,501505,1,0,2026-03-31
5845,PA,0096,05,90699,19907,522026,1,0,2026-03-31
6241,PA,0096,05,92550,19907,501022,1,0,2026-03-31
6242,PA,0096,05,92550,19907,523620,1,0,2026-03-31


In [278]:
PS_VAL_total_final_clientes_recomendados = (
    recomendaciones_para_salesforce.Cliente.nunique()
)
PS_VAL_total_final_clientes_recomendados

2129

In [279]:
PS_VAL_promedio_final_recomendaciones_por_cliente = (
    recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().mean()
)
PS_VAL_promedio_final_recomendaciones_por_cliente

2.0789102865194926

In [280]:
recomendaciones_para_salesforce.groupby("Sucursal")["Producto"].nunique()

Sucursal
01    45
02    39
03    46
04    35
05    36
Name: Producto, dtype: int64

In [281]:
PS_VAL_total_final_skus_recomendados = (
    recomendaciones_para_salesforce.Producto.nunique()
)
PS_VAL_total_final_skus_recomendados

56

**Conteo de marcas recomendadas**

In [282]:
count = (
    final_rec.groupby("id_cliente")["marca_rec"]
    .apply(lambda x: "-".join((x)))
    .reset_index(name="marcas_ordenadas")
)
count = (
    count.groupby("marcas_ordenadas")["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
)

In [283]:
PS_VAL_total_combinacion_rec_por_marca = count.shape[0]
PS_VAL_total_combinacion_rec_por_marca

94

**Conteo de SKUs recomendadas**

In [284]:
count2 = (
    recomendaciones_para_salesforce.groupby("Cliente")["Producto"]
    .apply(lambda x: "-".join(x.astype(str)))
    .reset_index(name="marcas_ordenadas")
)
count2 = (
    count2.groupby("marcas_ordenadas")["Cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
)
count2["f"] = count2["Cliente"] / count2["Cliente"].sum()
count2["fa"] = count2["f"].cumsum()

In [285]:
PS_VAL_total_combinacion_rec_por_sku = count2.shape[0]
PS_VAL_total_combinacion_rec_por_sku

997

In [286]:
PS_VAL_total_combinacion_rec_por_sku_al_80 = count2[count2["fa"] <= 0.8].shape[0]
PS_VAL_total_combinacion_rec_por_sku_al_80

571

In [287]:
recomendaciones_para_salesforce["Cliente"] = recomendaciones_para_salesforce[
    "Cliente"
].astype(str)
rutas_cliente["cod_cliente"] = rutas_cliente["cod_cliente"].astype(str)

In [288]:
PS_VAL_rutas_skus_clientes_por_sucursal = pd.merge(
    recomendaciones_para_salesforce,
    rutas_cliente,
    left_on="Cliente",
    right_on="cod_cliente",
    how="left",
)
PS_VAL_rutas_skus_clientes_por_sucursal = (
    PS_VAL_rutas_skus_clientes_por_sucursal.groupby("Sucursal")
    .agg({"cod_ruta": "nunique", "Cliente": "nunique", "Producto": "nunique"})
    .reset_index()
)
PS_VAL_rutas_skus_clientes_por_sucursal.columns = [
    "Sucursal",
    "cant_rutas",
    "cant_clientes",
    "cant_skus",
]
PS_VAL_rutas_skus_clientes_por_sucursal

,Sucursal,cant_rutas,cant_clientes,cant_skus
0,01,20,1037,45
1,02,10,460,39
2,03,8,425,46
3,04,3,145,35
4,05,2,81,36


In [289]:
print(f"RESUMEN PS para {fecha_tomorrow}")
print(f"*" * 20, "Metricas Iniciales", "*" * 20)
print(f"{'PS_VAL_total_clientes_venta:':<70}{PS_VAL_total_clientes_venta}")
print(f"{'PS_VAL_total_clientes_visita:':<70}{PS_VAL_total_clientes_visita}")
print(f"{'PS_VAL_total_clientes_visita_trad:':<70}{PS_VAL_total_clientes_visita_trad}")
print(f"{'PS_VAL_ultima_venta:':<70}{PS_VAL_ultima_venta}")
print(f"{'PS_VAL_ultima_visita:':<70}{PS_VAL_ultima_visita}")
print(f"{'PS_VAL_total_skus_venta:':<70}{PS_VAL_total_skus_venta}")
print(f"*" * 20, "Metricas del Proceso", "*" * 20)
print(f"{'PS_VAL_total_clientes_merged:':<70}{PS_VAL_total_clientes_merged}")
print(f"{'PS_VAL_total_fechas_unicas:':<70}{PS_VAL_total_fechas_unicas}")
print(f"{'PS_VAL_total_skus_merged:':<70}{PS_VAL_total_skus_merged}")
print(
    f"{'PS_VAL_total_clientes_imp_netovta_positivo:':<70}{PS_VAL_total_clientes_imp_netovta_positivo}"
)
print(
    f"{'PS_VAL_total_clientes_para_manhana:':<70}{PS_VAL_total_clientes_para_manhana}"
)
print(
    f"{'PS_VAL_total_skus_venta_para_manhana:':<70}{PS_VAL_total_skus_venta_para_manhana}"
)
# print(
#     f"{'PS_VAL_total_skus_recomendados_stock_y_activos:':<70}{PS_VAL_total_skus_recomendados_stock_y_activos}"
# )
# print(
#     f"{'PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda:':<70}{PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda}"
# )
print(f"*" * 20, "Metricas Finales", "*" * 20)
print(
    f"{'PS_VAL_total_final_skus_recomendados:':<70}{PS_VAL_total_final_skus_recomendados}"
)
print(
    f"{'PS_VAL_total_final_clientes_recomendados:':<70}{PS_VAL_total_final_clientes_recomendados}"
)
print(
    f"{'PS_VAL_promedio_final_recomendaciones_por_cliente:':<70}{PS_VAL_promedio_final_recomendaciones_por_cliente}"
)
print(
    f"{'PS_VAL_total_combinacion_rec_por_marca:':<70}{PS_VAL_total_combinacion_rec_por_marca}"
)
print(f"{'PS_VAL_marcas_recomendadas:':<70}{PS_VAL_marcas_recomendadas}")
print(
    f"{'PS_VAL_total_combinacion_rec_por_sku:':<70}{PS_VAL_total_combinacion_rec_por_sku}"
)
print(
    f"{'PS_VAL_total_combinacion_rec_por_sku_al_80:':<70}{PS_VAL_total_combinacion_rec_por_sku_al_80}"
)
print(
    "PS_VAL_rutas_skus_clientes_por_sucursal:\n",
    PS_VAL_rutas_skus_clientes_por_sucursal,
)

RESUMEN PS para 2026-03-31
******************** Metricas Iniciales ********************
PS_VAL_total_clientes_venta:                                          13105
PS_VAL_total_clientes_visita:                                         12453
PS_VAL_total_clientes_visita_trad:                                    12453
PS_VAL_ultima_venta:                                                  ['2026-03-28']
PS_VAL_ultima_visita:                                                 ['2026-03-28']
PS_VAL_total_skus_venta:                                              65
******************** Metricas del Proceso ********************
PS_VAL_total_clientes_merged:                                         12250
PS_VAL_total_fechas_unicas:                                           348
PS_VAL_total_skus_merged:                                             65
PS_VAL_total_clientes_imp_netovta_positivo:                           12250
PS_VAL_total_clientes_para_manhana:                                   2247
PS_V

In [290]:
recomendaciones_para_salesforce

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha
597,PA,0096,01,14344,19102,500713,1,0,2026-03-31
608,PA,0096,01,14355,19102,500742,1,0,2026-03-31
613,PA,0096,01,14356,19102,502501,1,0,2026-03-31
614,PA,0096,01,14356,19102,509671,1,0,2026-03-31
615,PA,0096,01,14356,19102,522450,1,0,2026-03-31
...,...,...,...,...,...,...,...,...,...
5844,PA,0096,05,90699,19907,501505,1,0,2026-03-31
5845,PA,0096,05,90699,19907,522026,1,0,2026-03-31
6241,PA,0096,05,92550,19907,501022,1,0,2026-03-31
6242,PA,0096,05,92550,19907,523620,1,0,2026-03-31


In [291]:
recomendaciones_para_salesforce.to_csv(
    f"Output/PS_todo_panama/D_base_pedidos_{fecha_tomorrow}.csv", index=False
)

In [292]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Panama/Output/PS_todo_panama/D_base_pedidos_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(recomendaciones_para_salesforce, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Panama/Output/PS_todo_panama/D_base_pedidos_2026-03-31.csv'],
 'partitions_values': {}}

### 5.8.3 Enviar Correo con Metricas de PS

In [293]:
# Define la zona horaria de Lima, Perú
lima_tz = pytz.timezone("America/Lima")

# Obtén la fecha y hora actual en la zona horaria de Lima
fecha_hoy_lima = datetime.now(lima_tz)
fecha_hoy = fecha_hoy_lima.strftime("%Y-%m-%d")
fecha_hoy

'2026-03-30'

In [294]:
# Resta un día para obtener la fecha de ayer
fecha_ayer_lima = fecha_hoy_lima - timedelta(days=1)

# Formatea la fecha en el formato año-mes-día
fecha_ayer = fecha_ayer_lima.strftime("%Y-%m-%d")
fecha_ayer

'2026-03-29'

In [295]:
# Obtener el día de la semana (1 es lunes, 7 es domingo)
dia_semana_hoy = (fecha_hoy_lima - timedelta(days=0)).weekday() + 1
dia_semana_hoy

1

In [296]:
# Si es lunes (1), restar 2 días para comparar
if dia_semana_hoy == 1:  # Lunes
    fecha_comparacion = fecha_ayer_lima - timedelta(days=2)
else:
    fecha_comparacion = fecha_ayer_lima

# Comprobar la fecha
if fecha_comparacion.strftime("%Y-%m-%d") == PS_VAL_ultima_venta[0]:
    mensaje_venta = f"Venta: PS listo. El archivo de ventas se encuentra actualizado hasta {fecha_comparacion.strftime('%Y-%m-%d')}, ya se puede ejecutar el modelo."
else:
    mensaje_venta = f"Venta: PS FALTA. Actualizado hasta {PS_VAL_ultima_venta[0]}"

In [297]:
mensaje_venta

'Venta: PS FALTA. Actualizado hasta 2026-03-28'

In [298]:
# Si es lunes (1), restar 2 días para comparar
if dia_semana_hoy == 1:  # Lunes
    fecha_comparacion = fecha_ayer_lima - timedelta(days=2)
else:
    fecha_comparacion = fecha_ayer_lima

# Comprobar la fecha
if fecha_comparacion.strftime("%Y-%m-%d") == PS_VAL_ultima_visita[0]:
    mensaje_visita = f"Visita: PS listo. El archivo de visitas se encuentra actualizado hasta {fecha_comparacion.strftime('%Y-%m-%d')}, ya se puede ejecutar el modelo."
else:
    mensaje_visita = f"Visita: PS FALTA. Actualizado hasta {PS_VAL_ultima_visita[0]}"

In [299]:
mensaje_visita

'Visita: PS FALTA. Actualizado hasta 2026-03-28'

In [300]:
# Credenciales para correo
remitente = "david.porta@ajegroup.com"
password = "pxqn kmuw drlq fvmv"

# Destinatario
asunto = f"Estado y Métricas PS para {fecha_tomorrow}"
# Lista de destinatarios
destinatarios = [
    "david.porta@ajegroup.com",
    "wilmer.rodriguez@ajegroup.com",
    "masaru.gonzales@ajegroup.com",
    "gherald.barzola@ajegroup.com",
    "jorge.delgado.pe@csoluciones.pe",
    "melissa.cotrina@ajegroup.com"
]
# destinatarios = ["david.porta@ajegroup.com"]

# Cuerpo de Correo
msg = MIMEMultipart()
msg["From"] = remitente
msg["To"] = ", ".join(destinatarios)
msg["Subject"] = asunto

In [301]:
# Convertir el DataFrame a HTML si PS_VAL_rutas_skus_clientes_por_sucursal es un DataFrame
tabla_rutas_skus_clientes = PS_VAL_rutas_skus_clientes_por_sucursal.to_html(index=False)

# quitar
# <tr><td>SKUs con stock y activos:</td><td>{PS_VAL_total_skus_recomendados_stock_y_activos}</td></tr>
# <tr><td>SKUs con stock, activos y precio según el lambda:</td><td>{PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda}</td></tr>

# Agregar mensaje al cuerpo del correo en formato HTML
cuerpo_correo = f"""
<html>
<head>
    <style>
        table {{
            width: 50%;
            border-collapse: collapse;
        }}
        th, td {{
            padding: 10px;
            border: 1px solid black;
            text-align: left;
        }}
        th {{
            background-color: #f2f2f2;
        }}
    </style>
</head>
<body>
    <h2>PANAMA</h2>
    <h3>Status de subida de PS Panama para {fecha_tomorrow}:</h3>
    <p>{mensaje_venta}</p>
    
    <h3>Pendientes:</h3>
    <p style="font-style: italic; color: #555;">Se pide apoyo a Masaru del Equipo Comercial para validar 
    que la subida de PS a Salesforce se realizó de manera satisfactoria.</p>

    <h3>RESUMEN PS Panama para {fecha_tomorrow}:</h3>

    <h4>Métricas de Input</h4>
    <table>
        <tr><td>Clientes en Historico de Ventas:</td><td>{PS_VAL_total_clientes_venta}</td></tr>
        <tr><td>Clientes en archivo de Visitas:</td><td>{PS_VAL_total_clientes_visita}</td></tr>
        <tr><td>Clientes en archivo de Visitas y Canal Tradicional:</td><td>{PS_VAL_total_clientes_visita_trad}</td></tr>
        <tr><td>Fecha ultima venta liquidada:</td><td>{PS_VAL_ultima_venta}</td></tr>
        <tr><td>Fecha ultima visita:</td><td>{PS_VAL_ultima_visita}</td></tr>
        <tr><td>SKUs distintos en Historico de Ventas:</td><td>{PS_VAL_total_skus_venta}</td></tr>
    </table>

    <h4>Métricas del Proceso</h4>
    <table>
        <tr><td>Clientes combinados de ventas y visitas:</td><td>{PS_VAL_total_clientes_merged}</td></tr>
        <tr><td>Fechas de liquidacion unicas en ventas:</td><td>{PS_VAL_total_fechas_unicas}</td></tr>
        <tr><td>SKUs distintos en ventas y visitas:</td><td>{PS_VAL_total_skus_merged}</td></tr>
        <tr><td>Clientes con importe de venta positivo:</td><td>{PS_VAL_total_clientes_imp_netovta_positivo}</td></tr>
        <tr><td>Clientes para mañana:</td><td>{PS_VAL_total_clientes_para_manhana}</td></tr>
        <tr><td>SKUs para clientes de mañana:</td><td>{PS_VAL_total_skus_venta_para_manhana}</td></tr>
    </table>

    <h4>Métricas Finales</h4>
    <table>
        <tr><td>SKUs finales recomendados:</td><td>{PS_VAL_total_final_skus_recomendados}</td></tr>
        <tr><td>Clientes finales recomendados:</td><td>{PS_VAL_total_final_clientes_recomendados}</td></tr>
        <tr><td>Clientes finales recomendados + clientes con visita y sin venta:</td><td>{PS_VAL_total_clientes_con_visita_no_venta}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego del modelo:</td><td>{PS_VAL_promedio_recs_por_cliente_modelo}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de stock:</td><td>{PS_VAL_promedio_recs_por_cliente_stock}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de reglas de negocio:</td><td>{PS_VAL_promedio_recs_por_cliente_negocio}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de reducir skus por segmento:</td><td>{PS_VAL_promedio_final_recomendaciones_por_cliente}</td></tr>
        <tr><td>Combinaciones de marca en recomendaciones:</td><td>{PS_VAL_total_combinacion_rec_por_marca}</td></tr>
        <tr><td>Marcas recomendadas:</td><td>{PS_VAL_marcas_recomendadas}</td></tr>
        <tr><td>Combinaciones de SKUs en recomendaciones:</td><td>{PS_VAL_total_combinacion_rec_por_sku}</td></tr>
        <tr><td>Combinaciones de SKUs al 80% de clientes:</td><td>{PS_VAL_total_combinacion_rec_por_sku_al_80}</td></tr>
    </table>

    <h4>Cantidad de Rutas, SKUs y Clientes por Sucursal:</h4>
    {tabla_rutas_skus_clientes}

</body>
</html>
"""

msg.attach(MIMEText(cuerpo_correo, "html"))

In [302]:
# Login SMTP de Gmail
server = smtplib.SMTP("smtp.gmail.com", 587)
server.starttls()
server.login(remitente, password)

(235, b'2.7.0 Accepted')

In [303]:
# Enviar correo
try:
    server.sendmail(remitente, destinatarios, msg.as_string())
    server.quit()
    print("Correo enviado")
except smtplib.SMTPException as e:
    print(f"Error SMTP al enviar correo: {e}")
except Exception as e:
    print(f"Error general al enviar correo: {e}")
except:
    print("Error al enviar correo")

Correo enviado


In [304]:
# # Detener el monitoreo
# running = False
# monitor_thread.join()

# # Graficar consumo de RAM y CPU
# fig, ax1 = plt.subplots(figsize=(10, 5))

# ax1.set_xlabel("Tiempo (segundos)")
# ax1.set_ylabel("RAM usada (GB)", color="tab:blue")
# ax1.plot(timestamps, mem_usage_gb, marker="o", linestyle="-", color="tab:blue", label="RAM (GB)")
# ax1.tick_params(axis="y", labelcolor="tab:blue")

# ax2 = ax1.twinx()  # Crear segundo eje Y para CPU
# ax2.set_ylabel("CPU usada (%)", color="tab:red")
# ax2.plot(timestamps, cpu_usage_percent, marker="x", linestyle="--", color="tab:red", label="CPU (%)")
# ax2.tick_params(axis="y", labelcolor="tab:red")

# fig.suptitle("Consumo de CPU y RAM del Notebook")
# ax1.grid()
# fig.tight_layout()
# plt.show()

In [305]:
# Apaga el Kernel para luego poder usar los demás notebooks sin llenar la memoria
import IPython
app = IPython.get_ipython()
app.kernel.do_shutdown(False)  # True indica apagar completamente el kernel

{'status': 'ok', 'restart': False}

In [306]:
#540 segundos